# Reply × LUISS - Transit Anomaly Detection
## Classical Pipeline vs Multi-Agent System

This notebook contains the full project end-to-end:

1. **Setup** — imports, paths, constants
2. **Preprocessing** — cleaning of the two raw datasets
3. **Feature engineering** — daily route table with traffic, segment, calendar and case features
4. **Classical Pipeline** — historical baselines, IsolationForest + LOF + Z-score, business rules, ranked report
5. **Multi-Agent System** — five-agent LangGraph workflow on a chosen airport perimeter
6. **Comparative Analysis** — same airport processed by both pipelines, with timing, overlap and final compared report

The dataset has no ground-truth anomaly labels, so the evaluation is operational rather than supervised — both pipelines produce a ranked alert queue, and the comparative analysis quantifies their agreement, performance and complementarity.

# 0. Setup

All imports, paths and reusable constants live here. The rest of the notebook references them without re-declaring.

In [120]:
# Standard library
import io
import json
import operator
import re
import time
import warnings
from pathlib import Path
from typing import TypedDict, Annotated, List

# Numerical and data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Modeling
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

# Multi-agent system
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

warnings.filterwarnings("ignore")

# Project configuration
# Centralised parameters make the notebook easier to reuse and tune.
PROJECT_CONFIG = {
    "random_state": 42,
    "contamination": 0.05,
    "min_rows_for_detection": 10,
    "entries_dev_ratio_threshold": 3,
    "seasonal_z_threshold": 2,
    "top_n_anomalies": 10,
}

# Reproducibility and shared thresholds
RANDOM_STATE = PROJECT_CONFIG["random_state"]
CONTAMINATION = PROJECT_CONFIG["contamination"]
MIN_ROWS_FOR_DETECTION = PROJECT_CONFIG["min_rows_for_detection"]
ENTRIES_DEV_RATIO_THRESHOLD = PROJECT_CONFIG["entries_dev_ratio_threshold"]
SEASONAL_Z_THRESHOLD = PROJECT_CONFIG["seasonal_z_threshold"]
TOP_N_ANOMALIES = PROJECT_CONFIG["top_n_anomalies"]

In [121]:
# Paths
BASE_DIR = Path.cwd()
IO_DIR = BASE_DIR / "io"
PREPROCESS_DIR = IO_DIR / "preprocessing"
FEAT_DIR = IO_DIR / "feat_engineering"
CLASSICAL_DIR = IO_DIR / "classical_report"
AGENT_DIR = IO_DIR / "agent_report"
COMPARED_DIR = IO_DIR / "compared_report"
IMAGES_DIR = CLASSICAL_DIR / "images"

for d in [IO_DIR, PREPROCESS_DIR, FEAT_DIR, CLASSICAL_DIR, AGENT_DIR, COMPARED_DIR, IMAGES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Raw inputs (place these two files in ./io/)
PASSENGERS_RAW = IO_DIR / "TIPOLOGIA_VIAGGIATORE.csv"
CASES_RAW = IO_DIR / "ALLARMI.csv"

# Intermediate outputs
PASSENGERS_CLEAN = PREPROCESS_DIR / "passenger_clean.csv"
CASES_CLEAN = PREPROCESS_DIR / "cases_clean.csv"
FEAT_OUT = FEAT_DIR / "feat_engineered.csv"

# Column translations and normalisation maps
COLUMN_MAPPING_PASSENGERS = {
    'AREOPORTO_ARRIVO': 'arrival_airport_code', 'AREOPORTO_PARTENZA': 'departure_airport_code',
    'DATA_PARTENZA': 'departure_date',
    'DESCR_AEREOPORTO_ARR': 'arrival_airport_name', 'DESCR_AEREOPORTO_PART': 'departure_airport_name',
    'CITTA_ARR': 'arrival_city', 'CITTA_PARTENZA': 'departure_city',
    'CODICE_PAESE_ARR': 'arrival_country_code', 'CODICE_PAESE_PART': 'departure_country_code',
    'PAESE_ARR': 'arrival_country', 'PAESE_PART': 'departure_country',
    'ZONA': 'zone',
    'ENTRATI': 'passengers_entries_count', 'INVESTIGATI': 'passengers_investigated_count',
    'ALLARMATI': 'passengers_flagged_count',
    'GENERE': 'gender', 'FLAG_TRANSITO': 'transit_flag', 'ESITO_CONTROLLO': 'control_result',
    'Tipo Documento': 'document_type', 'FASCIA ETA': 'age_range',
    '3nazionalita': 'nationality', 'compagnia%aerea': 'airline', 'num volo': 'flight_number'
}

COLUMN_MAPPING_CASES = {
    'OCCORRENZE': 'event_type',
    'AREOPORTO_ARRIVO': 'arrival_airport_code', 'AREOPORTO_PARTENZA': 'departure_airport_code',
    'DATA_PARTENZA': 'departure_date',
    'DESCR_AEREOPORTO_ARR': 'arrival_airport_name', 'DESCR_AEREOPORTO_PART': 'departure_airport_name',
    'CITTA_ARR': 'arrival_city_name', 'CITTA_PARTENZA': 'departure_city_name',
    'CODICE PAESE ARR': 'arrival_country_code', 'CODICE_PAESE_PART': 'departure_country_code',
    'MOTIVO_ALLARME': 'alarm_reason',
    'paese%arr': 'arrival_country_name', 'Paese Partenza': 'departure_country_name',
    'tot voli': 'total_flights', '3zona': 'region_zone'
}

GENDER_MAPPING = {
    'F': 'F', 'f': 'F', 'Femmina': 'F', 'Female': 'F', 'FEMALE': 'F', '2': 'F',
    'M': 'M', 'm': 'M', 'Maschio': 'M', 'Male': 'M', 'MALE': 'M', '1': 'M',
    'X': 'Other/NB', 'N/B': 'Other/NB'
}

COUNTRY_CODES = {'GB': 'GBR', 'EG': 'EGY', 'TR': 'TUR', 'AL': 'ALB', 'MA': 'MAR', 'AE': 'ARE'}

# Feature groups for the engineered table
ID_COLS = ["date", "route_city", "route_country", "route_airport"]
CALENDAR_COLS = ["year", "month", "day", "weekday", "is_weekend"]
BASE_COLS = ["entries", "investigated", "flagged", "investigation_rate", "flag_rate", "flag_given_investigated"]
SEGMENT_COLS = [
    "nationality_count", "avg_nat_entries", "max_nat_entries", "avg_nat_flag_rate", "max_nat_flag_rate",
    "document_type_count", "avg_doc_entries", "max_doc_entries", "avg_doc_flag_rate", "max_doc_flag_rate",
    "airline_count", "avg_airline_entries", "max_airline_entries", "avg_airline_flag_rate", "max_airline_flag_rate",
    "control_result_count", "avg_control_entries", "max_control_entries", "avg_control_flag_rate", "max_control_flag_rate"
]
CASE_COLS = ["has_case_match", "case_records", "total_flights", "unique_alarm_reasons", "unique_event_types", "alarm_density_per_entry"]
CHANGE_COLS = ['entries_lag1', 'entries_diff1', 'entries_pct_change1',
               'investigated_lag1', 'investigated_diff1', 'investigated_pct_change1',
               'flagged_lag1', 'flagged_diff1',
               'investigation_rate_lag1', 'investigation_rate_diff1', 'investigation_rate_pct_change1',
               'flag_rate_lag1', 'flag_rate_diff1']
VOLUME_FLAG_COLS = ["is_low_volume", "is_low_volume_50"]

## Shared utilities

Helper functions used by both pipelines: a robust 0-1 scoring with percentile clipping, and a cleaner that strips `<think>...</think>` tags emitted by reasoning-style local LLMs (phi4-mini-reasoning).

In [122]:
def robust_score(series, lower_q=0.01, upper_q=0.99):
    """Map a numeric signal to a 0-1 score using percentile clipping (less sensitive to outliers than min-max)."""
    s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if s.notna().sum() == 0:
        return pd.Series(0.0, index=s.index)
    lo, hi = s.quantile(lower_q), s.quantile(upper_q)
    if pd.isna(lo) or pd.isna(hi) or hi <= lo:
        return pd.Series(0.0, index=s.index)
    return ((s.clip(lo, hi) - lo) / (hi - lo)).fillna(0.0)

def clean_llm_output(text):
    """Remove <think>...</think> reasoning blocks emitted by some local models."""
    if not isinstance(text, str):
        return ""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

# 1. Preprocessing

The two raw CSVs (`TIPOLOGIA_VIAGGIATORE.csv` for passengers and `ALLARMI.csv` for security alarms) have inconsistent columns, mixed date formats and many placeholder values. This section harmonises them and saves clean intermediate files.

## 1.1 Passengers dataset

In [123]:
df_passengers = pd.read_csv(PASSENGERS_RAW)
print(f"Passengers raw: {df_passengers.shape}")
df_passengers.head()

Passengers raw: (5095, 33)


,NAZIONALITA,AREOPORTO_ARRIVO,AREOPORTO_PARTENZA,ANNO_PARTENZA,MESE_PARTENZA,GIORNO_PARTENZA,DATA_PARTENZA,DESCR_AEREOPORTO_ARR,DESCR_AEREOPORTO_PART,CITTA_ARR,...,COMPAGNIA_AEREA,NUMERO_VOLO,ESITO_CONTROLLO,note_operatore,codice_rischio,Tipo Documento,FASCIA ETA,3nazionalita,compagnia%aerea,num volo
0,ALB,NAP,DUR,2024,02,13,2024-02-13 07:30:00,Napoli Capodichino,King Shaka International,Napoli,...,Fly Dubai,FZ1681,RESPINTO,NaN,NaN,Passaporto,N.D.,ALB,Fly Dubai,FZ1681
1,NaN,FCO,JFK,2024,01,22,2024-01-22 16:35:00,Fiumicino,John F Kennedy International,Roma,...,ITA Airways,AZ0609,NaN,NaN,NaN,Carta d'identità,18-30,ALB,ITA Airways,AZ0609
2,ALB,TSF,TIA,2024,02,4,2024-02-04 20:10:00,Treviso-Sant'Angelo,Rinas Mother Teresa,Treviso,...,Ryanair DAC,FR8400,SEGNALATO,NaN,NaN,N.D.,31-45,ALB,Ryanair DAC,FR8400
3,AFG,FCO,IST,2024,01,25,2024-01-25 13:05:00,Fiumicino,Havalimani,Roma,...,Turkish Airlines,TK1865,NaN,NaN,NaN,N.D.,61+,AFG,Turkish Airlines,TK1865
4,ALB,BGY,MLE,2024,02,13,FEB 13 2024,Orio al Serio,Male International,Bergamo,...,Fly Dubai,FZ1571,SEGNALATO,NaN,NaN,Permesso di soggiorno,46-60,ALB,Fly Dubai,FZ1571


Five low-quality columns are dropped, placeholders are replaced with `UNKNOWN`, numeric counts are cleaned, and the logical chain `ENTRATI ≥ INVESTIGATI ≥ ALLARMATI` is enforced.

In [124]:
# Drop columns with too many inconsistencies
to_drop = ['FASCIA_ETA', 'TIPO_DOCUMENTO', 'NAZIONALITA', 'NUMERO_VOLO', 'COMPAGNIA_AEREA',
           'ANNO_PARTENZA', 'MESE_PARTENZA', 'GIORNO_PARTENZA', 'note_operatore', 'codice_rischio']
df_passengers.drop(columns=to_drop, errors='ignore', inplace=True)

# Standardise gender
df_passengers['GENERE'] = df_passengers['GENERE'].map(GENDER_MAPPING).fillna('UNKNOWN')

# Clean numeric passenger counts
for col in ['ENTRATI', 'INVESTIGATI', 'ALLARMATI']:
    df_passengers[col] = (df_passengers[col].astype(str)
                          .str.replace('pax', '', case=False)
                          .str.replace('~', '').str.replace(',', '.'))
    df_passengers[col] = pd.to_numeric(df_passengers[col], errors='coerce').astype('Int64')

df_passengers = df_passengers.dropna(subset=['ENTRATI', 'INVESTIGATI'], how='all').copy()

# Negatives -> 0
for col in ['INVESTIGATI', 'ALLARMATI', 'ENTRATI']:
    df_passengers.loc[df_passengers[col] < 0, col] = 0

# Recover missing values via the logical chain
df_passengers['ALLARMATI'].fillna(0, inplace=True)
df_passengers['INVESTIGATI'] = df_passengers['INVESTIGATI'].fillna(df_passengers['ALLARMATI'])
df_passengers['ENTRATI'] = df_passengers['ENTRATI'].fillna(df_passengers['INVESTIGATI'])

# Enforce ENTRATI >= INVESTIGATI >= ALLARMATI and a realistic upper bound
valid = ((df_passengers['ALLARMATI'] <= df_passengers['INVESTIGATI']) &
         (df_passengers['INVESTIGATI'] <= df_passengers['ENTRATI']) &
         (df_passengers['ENTRATI'] <= 853))
df_passengers = df_passengers[valid].copy()

Departure dates use mixed formats and Italian month abbreviations.

In [125]:
df_passengers['DATA_PARTENZA'] = df_passengers['DATA_PARTENZA'].str.upper().str.replace('GEN', 'JAN')
df_passengers['DATA_PARTENZA'] = pd.to_datetime(df_passengers['DATA_PARTENZA'], format='mixed', dayfirst=True)

Country codes are converted to 3-letter ISO format. Text columns are uppercased and column names are translated to English.

In [126]:
# Replace placeholders
placeholders = ['N.D.', 'n.d.', 'N/A', ' ', '?', '//']
df_passengers = df_passengers.replace(placeholders, 'UNKNOWN')
df_passengers['ESITO_CONTROLLO'] = df_passengers['ESITO_CONTROLLO'].fillna('UNKNOWN')
df_passengers = df_passengers.drop_duplicates()

# ISO country codes
df_passengers['CODICE_PAESE_ARR'] = df_passengers['CODICE_PAESE_ARR'].replace('IT', 'ITA')
df_passengers['CODICE_PAESE_PART'].replace(COUNTRY_CODES, inplace=True)

# Uppercase text columns
for col in ['AREOPORTO_ARRIVO', 'AREOPORTO_PARTENZA', 'CITTA_ARR', 'CITTA_PARTENZA',
            'PAESE_ARR', 'PAESE_PART', 'DESCR_AEREOPORTO_ARR', 'DESCR_AEREOPORTO_PART']:
    df_passengers[col] = df_passengers[col].str.upper()

# English column names
df_passengers.rename(columns=COLUMN_MAPPING_PASSENGERS, inplace=True)
df_passengers.head()

,arrival_airport_code,departure_airport_code,departure_date,arrival_airport_name,departure_airport_name,arrival_city,departure_city,arrival_country_code,departure_country_code,arrival_country,...,passengers_investigated_count,passengers_flagged_count,gender,transit_flag,control_result,document_type,age_range,nationality,airline,flight_number
0,NAP,DUR,2024-02-13 07:30:00,NAPOLI CAPODICHINO,KING SHAKA INTERNATIONAL,NAPOLI,DURBAN,ITA,ZAF,ITALIA,...,1,0,F,Singola Tratta,RESPINTO,Passaporto,UNKNOWN,ALB,Fly Dubai,FZ1681
2,TSF,TIA,2024-04-02 20:10:00,TREVISO-SANT'ANGELO,RINAS MOTHER TERESA,TREVISO,TIRANA,ITA,ALB,ITALIA,...,58,13,F,Singola Tratta,SEGNALATO,UNKNOWN,31-45,ALB,Ryanair DAC,FR8400
3,FCO,IST,2024-01-25 13:05:00,FIUMICINO,HAVALIMANI,ROMA,ISTANBUL,ITA,TR,ITALIA,...,1,0,M,Singola Tratta,UNKNOWN,UNKNOWN,61+,AFG,Turkish Airlines,TK1865
4,BGY,MLE,2024-02-13 00:00:00,ORIO AL SERIO,MALE INTERNATIONAL,BERGAMO,MALE,ITA,MDV,ITALIA,...,2,1,F,Singola Tratta,SEGNALATO,Permesso di soggiorno,46-60,ALB,Fly Dubai,FZ1571
5,TSF,LTN,2024-02-18 16:30:00,TREVISO-SANT'ANGELO,LONDON LUTON,TREVISO,LONDRA,ITA,GBR,ITALIA,...,3,0,F,Singola Tratta,RESPINTO,Carta d'identità,18-30,ALB,Ryanair DAC,FR1050


## 1.2 Cases dataset

In [127]:
df_cases = pd.read_csv(CASES_RAW)
print(f"Cases raw: {df_cases.shape}")
df_cases.head()

Cases raw: (5080, 24)


,OCCORRENZE,AREOPORTO_ARRIVO,AREOPORTO_PARTENZA,ANNO_PARTENZA,MESE_PARTENZA,DATA_PARTENZA,DESCR_AEREOPORTO_ARR,DESCR_AEREOPORTO_PART,CITTA_ARR,CITTA_PARTENZA,...,ZONA,TOT,MOTIVO_ALLARME,note_operatore,flag_rischio,Paese Partenza,CODICE PAESE ARR,3zona,paese%arr,tot voli
0,Voli con Allarmi,FCO,IST,2024,01,2024-01-30 09:15:00,Fiumicino,Havalimani,Roma,Istanbul,...,5,1,Manuale,NaN,NaN,Turchia,ITA,5,Italia,1
1,Viaggiatori con Allarmi,CIA,STN,2024,02,2024-02-03 13:15:00,Ciampino,Stansted,Roma,Londra,...,5,5,Manuale,NaN,NaN,Regno Unito,ITA,5,Italia,5
2,Viaggiatori entrati nel Sistema,FCO,LHR,2024,01,2024-01-15 08:45:00,Fiumicino,London Heathrow,Roma,Londra,...,5,110,TSC,NaN,NaN,Regno Unito,ITA,5,Italia,110
3,Voli con Allarmi,MXP,LHR,2024,02,2024-02-02 08:40:00,Malpensa,London Heathrow,Milano,Londra,...,2,1,SDI,NaN,NaN,Regno Unito,ITA,2,Italia,1
4,Viaggiatori con Allarmi,PSA,BRS,2024,02,2024-02-16 12:50:00,Galileo Galilei,Bristol,Pisa,Bristol,...,8,2,INTERPOL,NaN,NaN,Regno Unito,ITA,8,Italia,2


Inconsistent or redundant columns are dropped, then placeholders, dates and uppercase formatting are handled in one pass.

In [128]:
cols_drop = ['PAESE_ARR', 'PAESE_PART', 'ZONA', 'note_operatore', 'flag_rischio',
             'ANNO_PARTENZA', 'MESE_PARTENZA', 'TOT', 'CODICE_PAESE_ARR']
df_cases.drop(columns=cols_drop, errors='ignore', inplace=True)
df_cases = df_cases.drop_duplicates()

# Numeric flights
df_cases['tot voli'] = df_cases['tot voli'].astype(str).str.extract(r'(\d+)').astype(float).fillna(0).astype(int)

# Dates
df_cases['DATA_PARTENZA'] = df_cases['DATA_PARTENZA'].astype(str).str.upper().str.replace('GEN', 'JAN')
df_cases['DATA_PARTENZA'] = pd.to_datetime(df_cases['DATA_PARTENZA'], format='mixed', dayfirst=True, errors='coerce')

# Uppercase text columns
for col in ['AREOPORTO_ARRIVO', 'AREOPORTO_PARTENZA', 'DESCR_AEREOPORTO_ARR', 'DESCR_AEREOPORTO_PART']:
    df_cases[col] = df_cases[col].str.upper()

# Replace placeholders in OCCORRENZE
df_cases['OCCORRENZE'] = df_cases['OCCORRENZE'].replace(['???', 'N/C', 'Altro'], 'UNKNOWN')

# Departure airport name fallback
ph = [' ', '?', 'N.D.', 'ND', '-', '//', 'N/A ', '  ', '- ', 'N/A', '', 'NULL']
df_cases['DESCR_AEREOPORTO_PART'] = df_cases['DESCR_AEREOPORTO_PART'].str.upper().str.strip()
df_cases['DESCR_AEREOPORTO_PART'] = df_cases['DESCR_AEREOPORTO_PART'].replace(ph, np.nan)
df_cases['DESCR_AEREOPORTO_PART'] = df_cases['DESCR_AEREOPORTO_PART'].fillna(df_cases['AREOPORTO_PARTENZA'])

# Departure country code fallback
ph_cc = ['n.d.', 'ND', '?', 'unknown', 'EU', 'XX', '00', '//', '-', 'ZZ', ' ']
df_cases['CODICE_PAESE_PART'] = df_cases['CODICE_PAESE_PART'].replace(COUNTRY_CODES)
df_cases['CODICE_PAESE_PART'] = df_cases['CODICE_PAESE_PART'].replace(ph_cc, 'UNKNOWN')
df_cases['CODICE_PAESE_PART'].fillna('UNKNOWN', inplace=True)
df_cases['CODICE_PAESE_PART'] = df_cases['CODICE_PAESE_PART'].str.upper()

ref_map = (df_cases[df_cases['CODICE_PAESE_PART'] != 'UNKNOWN']
           .set_index('Paese Partenza')['CODICE_PAESE_PART'].to_dict())
df_cases.loc[df_cases['CODICE_PAESE_PART'] == 'UNKNOWN', 'CODICE_PAESE_PART'] = (
    df_cases['Paese Partenza'].map(ref_map))

# Departure city
df_cases['CITTA_PARTENZA'].fillna('unknown', inplace=True)
df_cases['CITTA_PARTENZA'] = df_cases.apply(
    lambda r: f"UNKNOWN ({r['Paese Partenza']})" if r['CITTA_PARTENZA'] in ph_cc else r['CITTA_PARTENZA'], axis=1)

# Final uppercase pass
for col in ['Paese Partenza', 'CITTA_ARR', 'CITTA_PARTENZA', 'paese%arr', 'MOTIVO_ALLARME']:
    df_cases[col] = df_cases[col].str.upper()
df_cases['MOTIVO_ALLARME'].fillna('UNKNOWN', inplace=True)

0        MANUALE
1        MANUALE
2            TSC
3            SDI
4       INTERPOL
          ...   
5075         TSC
5076         TSC
5077        NSIS
5078         TSC
5079         TSC
Name: MOTIVO_ALLARME, Length: 5030, dtype: str

A country must map to a single region — we enforce this using the modal mapping per country.

In [129]:
country_zone = df_cases.groupby('CODICE_PAESE_PART')['3zona'].agg(lambda x: x.mode().iloc[0]).to_dict()
df_cases['3zona'] = df_cases['CODICE_PAESE_PART'].map(country_zone)
df_cases.rename(columns=COLUMN_MAPPING_CASES, inplace=True)
df_cases.head()

,event_type,arrival_airport_code,departure_airport_code,departure_date,arrival_airport_name,departure_airport_name,arrival_city_name,departure_city_name,departure_country_code,alarm_reason,departure_country_name,arrival_country_code,region_zone,arrival_country_name,total_flights
0,Voli con Allarmi,FCO,IST,2024-01-30 09:15:00,FIUMICINO,HAVALIMANI,ROMA,ISTANBUL,TUR,MANUALE,TURCHIA,ITA,5.0,ITALIA,1
1,Viaggiatori con Allarmi,CIA,STN,2024-03-02 13:15:00,CIAMPINO,STANSTED,ROMA,LONDRA,GBR,MANUALE,REGNO UNITO,ITA,2.0,ITALIA,5
2,Viaggiatori entrati nel Sistema,FCO,LHR,2024-01-15 08:45:00,FIUMICINO,LONDON HEATHROW,ROMA,LONDRA,GBR,TSC,REGNO UNITO,ITA,2.0,ITALIA,110
3,Voli con Allarmi,MXP,LHR,2024-02-02 08:40:00,MALPENSA,LONDON HEATHROW,MILANO,LONDRA,GBR,SDI,REGNO UNITO,ITA,2.0,ITALIA,1
4,Viaggiatori con Allarmi,PSA,BRS,2024-02-16 12:50:00,GALILEO GALILEI,BRISTOL,PISA,BRISTOL,GBR,INTERPOL,REGNO UNITO,ITA,2.0,ITALIA,2


Save the cleaned datasets — the multi-agent pipeline reads them back later from disk.

In [130]:
df_passengers.to_csv(PASSENGERS_CLEAN, index=False)
df_cases.to_csv(CASES_CLEAN, index=False)
print(f"Passengers clean: {df_passengers.shape} -> {PASSENGERS_CLEAN}")
print(f"Cases clean: {df_cases.shape} -> {CASES_CLEAN}")

Passengers clean: (4533, 23) -> c:\Users\AFGWO\OneDrive\Desktop\Lectures\Machine Learning\reply-classical-vs-multiagent\src\io\preprocessing\passenger_clean.csv
Cases clean: (5030, 15) -> c:\Users\AFGWO\OneDrive\Desktop\Lectures\Machine Learning\reply-classical-vs-multiagent\src\io\preprocessing\cases_clean.csv


# 2. Feature Engineering

We build a daily route table aggregated at `date × route_airport`. This grain is unique, aligns with operational behavior and gives the most precise level for anomaly detection. `route_city` and `route_country` are kept for interpretability.

In [131]:
passenger = df_passengers.copy()
cases = df_cases.copy()

## 2.1 Calendar and route features

In [132]:
passenger["departure_date"] = pd.to_datetime(passenger["departure_date"], errors="coerce")
cases["departure_date"] = pd.to_datetime(cases["departure_date"], errors="coerce")

for d in [passenger, cases]:
    d["year"] = d["departure_date"].dt.year
    d["month"] = d["departure_date"].dt.month
    d["day"] = d["departure_date"].dt.day
    d["weekday"] = d["departure_date"].dt.dayofweek
    d["is_weekend"] = d["weekday"].isin([5, 6]).astype(int)
    d["date"] = d["departure_date"].dt.floor("D")

passenger["document_type"] = passenger["document_type"].fillna("UNKNOWN")
cases["alarm_reason"] = cases["alarm_reason"].fillna("UNKNOWN")

passenger["route_country"] = passenger["departure_country_code"].astype(str) + "_" + passenger["arrival_country_code"].astype(str)
passenger["route_city"] = passenger["departure_city"].astype(str) + "_" + passenger["arrival_city"].astype(str)
passenger["route_airport"] = passenger["departure_airport_code"].astype(str) + "_" + passenger["arrival_airport_code"].astype(str)

cases["route_country"] = cases["departure_country_code"].astype(str) + "_" + cases["arrival_country_code"].astype(str)
cases["route_city"] = cases["departure_city_name"].astype(str) + "_" + cases["arrival_city_name"].astype(str)
cases["route_airport"] = cases["departure_airport_code"].astype(str) + "_" + cases["arrival_airport_code"].astype(str)

## 2.2 Ratio features

Investigation rate, flag rate and conditional flag rate are computed at the row level. A low-volume flag is added because many rows have very few entries (so we filter without dropping).

In [133]:
passenger["is_low_volume"] = (passenger["passengers_entries_count"] < 10).astype(int)
passenger["investigation_rate"] = np.where(passenger["passengers_entries_count"] > 0,
    passenger["passengers_investigated_count"] / passenger["passengers_entries_count"], np.nan)
passenger["flag_rate"] = np.where(passenger["passengers_entries_count"] > 0,
    passenger["passengers_flagged_count"] / passenger["passengers_entries_count"], np.nan)
passenger["flag_given_investigated"] = np.where(passenger["passengers_investigated_count"] > 0,
    passenger["passengers_flagged_count"] / passenger["passengers_investigated_count"], np.nan)

## 2.3 Base daily route table

In [134]:
route_daily = (passenger.groupby(["date", "route_airport"], as_index=False)
    .agg(route_city=("route_city", "first"),
         route_country=("route_country", "first"),
         entries=("passengers_entries_count", "sum"),
         investigated=("passengers_investigated_count", "sum"),
         flagged=("passengers_flagged_count", "sum")))

route_daily["investigation_rate"] = np.where(route_daily["entries"] > 0,
    route_daily["investigated"] / route_daily["entries"], np.nan)
route_daily["flag_rate"] = np.where(route_daily["entries"] > 0,
    route_daily["flagged"] / route_daily["entries"], np.nan)
route_daily["flag_given_investigated"] = np.where(route_daily["investigated"] > 0,
    route_daily["flagged"] / route_daily["investigated"], np.nan)

route_daily["year"] = route_daily["date"].dt.year
route_daily["month"] = route_daily["date"].dt.month
route_daily["day"] = route_daily["date"].dt.day
route_daily["weekday"] = route_daily["date"].dt.dayofweek
route_daily["is_weekend"] = route_daily["weekday"].isin([5, 6]).astype(int)
route_daily.head()

,date,route_airport,route_city,route_country,entries,investigated,flagged,investigation_rate,flag_rate,flag_given_investigated,year,month,day,weekday,is_weekend
0,2023-12-31,JFK_FCO,NEW YORK_ROMA,USA_ITA,3,3,0,1.0,0.0,0.0,2023,12,31,6,1
1,2024-01-01,ADB_MXP,SMIRNE_MILANO,TUR_ITA,1,1,0,1.0,0.0,0.0,2024,1,1,0,0
2,2024-01-01,AMM_FCO,AMMAN_ROMA,JOR_ITA,2,0,0,0.0,0.0,NaN,2024,1,1,0,0
3,2024-01-01,BEG_BGY,BELGRADO_BERGAMO,SRB_ITA,1,1,0,1.0,0.0,0.0,2024,1,1,0,0
4,2024-01-01,BFS_BGY,BELFAST_BERGAMO,GBR_ITA,1,1,0,1.0,0.0,0.0,2024,1,1,0,0


## 2.4 Segment-level features

For each route-day, we aggregate by nationality, document type, airline and control result. Per segment we compute count (diversity), average and maximum entries, and average/maximum flag rate.

In [135]:
# Nationality
nat = (passenger.groupby(["date", "route_airport", "nationality"], as_index=False)
    .agg(nat_entries=("passengers_entries_count", "sum"),
         nat_flagged=("passengers_flagged_count", "sum")))
nat["nat_flag_rate"] = np.where(nat["nat_entries"] > 0, nat["nat_flagged"] / nat["nat_entries"], np.nan)
nat_features = (nat.groupby(["date", "route_airport"], as_index=False)
    .agg(nationality_count=("nationality", "nunique"),
         avg_nat_entries=("nat_entries", "mean"),
         max_nat_entries=("nat_entries", "max"),
         avg_nat_flag_rate=("nat_flag_rate", "mean"),
         max_nat_flag_rate=("nat_flag_rate", "max")))
route_daily = route_daily.merge(nat_features, on=["date", "route_airport"], how="left")

# Document type
doc = (passenger.groupby(["date", "route_airport", "document_type"], as_index=False)
    .agg(doc_entries=("passengers_entries_count", "sum"),
         doc_flagged=("passengers_flagged_count", "sum")))
doc["doc_flag_rate"] = np.where(doc["doc_entries"] > 0, doc["doc_flagged"] / doc["doc_entries"], np.nan)
doc_features = (doc.groupby(["date", "route_airport"], as_index=False)
    .agg(document_type_count=("document_type", "nunique"),
         avg_doc_entries=("doc_entries", "mean"),
         max_doc_entries=("doc_entries", "max"),
         avg_doc_flag_rate=("doc_flag_rate", "mean"),
         max_doc_flag_rate=("doc_flag_rate", "max")))
route_daily = route_daily.merge(doc_features, on=["date", "route_airport"], how="left")

# Airline
air = (passenger.groupby(["date", "route_airport", "airline"], as_index=False)
    .agg(airline_entries=("passengers_entries_count", "sum"),
         airline_flagged=("passengers_flagged_count", "sum")))
air["airline_flag_rate"] = np.where(air["airline_entries"] > 0, air["airline_flagged"] / air["airline_entries"], np.nan)
air_features = (air.groupby(["date", "route_airport"], as_index=False)
    .agg(airline_count=("airline", "nunique"),
         avg_airline_entries=("airline_entries", "mean"),
         max_airline_entries=("airline_entries", "max"),
         avg_airline_flag_rate=("airline_flag_rate", "mean"),
         max_airline_flag_rate=("airline_flag_rate", "max")))
route_daily = route_daily.merge(air_features, on=["date", "route_airport"], how="left")

# Control result
ctr = (passenger.groupby(["date", "route_airport", "control_result"], dropna=False, as_index=False)
    .agg(control_entries=("passengers_entries_count", "sum"),
         control_flagged=("passengers_flagged_count", "sum")))
ctr["control_flag_rate"] = np.where(ctr["control_entries"] > 0, ctr["control_flagged"] / ctr["control_entries"], np.nan)
ctr_features = (ctr.groupby(["date", "route_airport"], as_index=False)
    .agg(control_result_count=("control_result", "nunique"),
         avg_control_entries=("control_entries", "mean"),
         max_control_entries=("control_entries", "max"),
         avg_control_flag_rate=("control_flag_rate", "mean"),
         max_control_flag_rate=("control_flag_rate", "max")))
route_daily = route_daily.merge(ctr_features, on=["date", "route_airport"], how="left")

## 2.5 Case features

Cases data is aggregated per route-day and merged into the main table. `has_case_match` is set before fillna so we preserve the information that some routes have no matched alarm cases.

In [136]:
cases_daily = (cases.groupby(["date", "route_airport"], as_index=False)
    .agg(case_records=("event_type", "size"),
         total_flights=("total_flights", "sum"),
         unique_alarm_reasons=("alarm_reason", "nunique"),
         unique_event_types=("event_type", "nunique")))

route_daily = route_daily.merge(cases_daily, on=["date", "route_airport"], how="left")
route_daily["has_case_match"] = route_daily["case_records"].notna().astype(int)
route_daily["alarm_density_per_entry"] = np.where(route_daily["entries"] > 0,
    route_daily["case_records"] / route_daily["entries"], np.nan)

route_daily = route_daily.sort_values(["route_airport", "date"]).reset_index(drop=True)

## 2.6 Temporal change features

For each route, we compute lag-1, absolute difference and percentage change of entries, investigations, flags and rates. These capture day-to-day shifts in operational behavior.

In [137]:
for col in ["entries", "investigated", "flagged", "investigation_rate", "flag_rate"]:
    route_daily[f"{col}_lag1"] = route_daily.groupby("route_airport")[col].shift(1)
    route_daily[f"{col}_diff1"] = route_daily[col] - route_daily[f"{col}_lag1"]
    route_daily[f"{col}_pct_change1"] = np.where(
        route_daily[f"{col}_lag1"].notna() & (route_daily[f"{col}_lag1"] != 0),
        (route_daily[col] - route_daily[f"{col}_lag1"]) / route_daily[f"{col}_lag1"], np.nan)

## 2.7 Sanity check + final cleanup

We plot the top 3 routes to make sure the temporal patterns look sensible, then handle the remaining structural issues.

In [138]:
top_routes = route_daily.groupby("route_city")["entries"].sum().sort_values(ascending=False).head(3).index.tolist()

for route in top_routes:
    tmp = route_daily[route_daily["route_city"] == route].sort_values("date")
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
    axes[0].plot(tmp["date"], tmp["entries"]); axes[0].set_title(f"Entries — {route}")
    axes[1].plot(tmp["date"], tmp["flag_rate"]); axes[1].set_title(f"Flag rate — {route}")
    for ax in axes: ax.tick_params(axis="x", rotation=45)
    plt.tight_layout(); plt.show()

In [139]:
# Distribution of entries
plt.figure(figsize=(8, 4))
route_daily["entries"].dropna().hist(bins=50)
plt.title("Distribution of entries"); plt.xlabel("Entries"); plt.ylabel("Frequency")
plt.tight_layout(); plt.show()

route_daily["is_low_volume"] = (route_daily["entries"] < 10).astype(int)
route_daily["is_low_volume_50"] = (route_daily["entries"] < 50).astype(int)

# Zero-fill case columns and re-derive density
for c in ["case_records", "total_flights", "unique_alarm_reasons", "unique_event_types"]:
    route_daily[c] = route_daily[c].fillna(0)
route_daily["alarm_density_per_entry"] = np.where(route_daily["entries"] > 0,
    route_daily["case_records"] / route_daily["entries"], np.nan)

# Two pct-change columns are >50% missing — drop
route_daily.drop(columns=['flagged_pct_change1', 'flag_rate_pct_change1'], inplace=True, errors='ignore')

# Final feature table
full_cols = [c for c in (ID_COLS + CALENDAR_COLS + BASE_COLS + SEGMENT_COLS + CASE_COLS + CHANGE_COLS + VOLUME_FLAG_COLS) if c in route_daily.columns]
feat = route_daily[full_cols].copy()
feat.to_csv(FEAT_OUT, index=False)
print(f"Feature-engineered table: {feat.shape} -> {FEAT_OUT}")

Feature-engineered table: (3267, 56) -> c:\Users\AFGWO\OneDrive\Desktop\Lectures\Machine Learning\reply-classical-vs-multiagent\src\io\feat_engineering\feat_engineered.csv


# 3. Classical Pipeline

Unsupervised anomaly detection on the full dataset. The dataset has no ground-truth anomaly labels, so we do not use a train/test split and we do not report supervised metrics — that would only simulate validation without real labels. Instead we evaluate operational criteria: anomaly rate, agreement between detectors, stability under different contamination thresholds, and interpretability of the final alert queue.

We start the **classical timer** here. It runs through the end of the classical export, and is reported at the bottom of the notebook side by side with the multi-agent timer.

In [140]:
CLASSICAL_START = time.time()
df = pd.read_csv(FEAT_OUT)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["route_airport", "date"]).reset_index(drop=True)
print(f"Loaded {len(df)} rows, {df['route_airport'].nunique()} routes")

Loaded 3267 rows, 514 routes


## 3.1 Historical baselines

Two baselines define what "normal" looks like per route:

1. **Rolling averages** (7-obs and 30-obs windows) capture recent trends
2. **Monthly seasonal baselines** capture recurring monthly patterns

We use observation-based windows rather than calendar-based: the dataset is irregular (median 3-day gap, some routes month-long). A calendar-based 7-day window would often contain only 1-2 data points.

In [141]:
for col in ["entries", "flag_rate", "investigation_rate"]:
    df[f"{col}_roll7"] = df.groupby("route_airport")[col].transform(lambda s: s.rolling(7, min_periods=3).mean())
    df[f"{col}_roll30"] = df.groupby("route_airport")[col].transform(lambda s: s.rolling(30, min_periods=7).mean())
    df[f"{col}_dev_ratio7"] = np.where(
        df[f"{col}_roll7"].notna() & (df[f"{col}_roll7"] > 0),
        df[col] / df[f"{col}_roll7"], np.nan)

print(f"Rolling 7-obs coverage: {df['entries_roll7'].notna().mean():.1%}")
print(f"Rolling 30-obs coverage: {df['entries_roll30'].notna().mean():.1%}")

Rolling 7-obs coverage: 75.4%
Rolling 30-obs coverage: 55.4%


### Monthly seasonal baselines

Standard `seasonal_decompose` requires regularly-spaced daily data. Our dataset is too irregular for it. Instead, for each `route_airport × month` we compute the average entries as a baseline, then the residual normalised to z-score per route. A static route-level mean is also computed as a stable reference, useful for routes with too few observations to support rolling windows.

In [142]:
df["entries_monthly_baseline"] = df.groupby(["route_airport", "month"])["entries"].transform("mean")
df["entries_residual"] = df["entries"] - df["entries_monthly_baseline"]

residual_std = df.groupby("route_airport")["entries_residual"].transform("std")
df["entries_residual_z"] = np.where(
    residual_std.notna() & (residual_std > 0),
    df["entries_residual"] / residual_std, np.nan)

df["entries_route_mean"] = df.groupby("route_airport")["entries"].transform("mean")
df["entries_vs_route_mean"] = np.where(df["entries_route_mean"] > 0,
    df["entries"] / df["entries_route_mean"], np.nan)

print(f"Monthly baseline coverage: {df['entries_monthly_baseline'].notna().mean():.1%}")
print(f"Residual z-score coverage: {df['entries_residual_z'].notna().mean():.1%}")

Monthly baseline coverage: 100.0%
Residual z-score coverage: 79.6%


### Sanity check on baselines

In [143]:
top3 = df.groupby("route_city")["entries"].sum().sort_values(ascending=False).head(3).index.tolist()

for route in top3:
    tmp = df[df["route_city"] == route].sort_values("date")
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
    axes[0].plot(tmp["date"], tmp["entries"], alpha=0.5, label="entries")
    axes[0].plot(tmp["date"], tmp["entries_roll7"], linewidth=2, label="rolling 7-obs")
    axes[0].set_title(f"Entries vs Rolling Baseline — {route}"); axes[0].legend()
    axes[1].bar(tmp["date"], tmp["entries_residual"], alpha=0.7)
    axes[1].axhline(0, linewidth=0.5)
    axes[1].set_title(f"Monthly Seasonal Residual — {route}")
    for ax in axes: ax.tick_params(axis="x", rotation=45)
    plt.tight_layout(); plt.show()

## 3.2 Missing value handling

Different column groups have different reasons for missing values, so we apply targeted strategies — never blanket fillna across all columns.

In [144]:
change_fill = [c for c in df.columns if "_lag1" in c or "_diff1" in c or "_pct_change1" in c]
df[change_fill] = df[change_fill].fillna(0)

df["flag_given_investigated"] = df["flag_given_investigated"].fillna(0)

rate_fill = ["investigation_rate", "flag_rate", "alarm_density_per_entry",
             "avg_nat_flag_rate", "max_nat_flag_rate",
             "avg_doc_flag_rate", "max_doc_flag_rate",
             "avg_airline_flag_rate", "max_airline_flag_rate",
             "avg_control_flag_rate", "max_control_flag_rate"]
df[rate_fill] = df[rate_fill].fillna(0)

rolling_fill = [c for c in df.columns if "_roll" in c or "_dev_ratio" in c]
df[rolling_fill] = df[rolling_fill].fillna(df[rolling_fill].median())

for c in ["entries_residual", "entries_residual_z", "entries_monthly_baseline",
          "entries_route_mean", "entries_vs_route_mean"]:
    df[c] = df[c].fillna(0)

remaining = df.select_dtypes(include=[np.number]).isna().sum()
remaining = remaining[remaining > 0]
print("All numeric columns complete." if len(remaining) == 0 else f"Remaining NaN: {len(remaining)} columns")

All numeric columns complete.


## 3.3 Detection vs reporting features

We split features by causal role:

- **Detection features** describe traffic behavior and historical deviations — they exist before any operational intervention
- **Reporting features** describe consequences (flag rates, investigation outcomes, alarm cases) — using them to detect anomalies would create leakage

The detection layer uses only detection features. Reporting features are added back during post-processing to explain anomalies in operational terms.

In [145]:
DETECTION_FEATURES = [
    "entries",
    "entries_lag1", "entries_diff1", "entries_pct_change1",
    "entries_roll7", "entries_roll30", "entries_dev_ratio7",
    "entries_monthly_baseline", "entries_residual", "entries_residual_z",
    "entries_route_mean", "entries_vs_route_mean",
    "weekday", "is_weekend",
    "is_low_volume", "is_low_volume_50",
]

REPORTING_FEATURES = [
    "investigated", "flagged",
    "investigation_rate", "flag_rate", "flag_given_investigated",
    "has_case_match", "case_records", "total_flights",
    "unique_alarm_reasons", "unique_event_types", "alarm_density_per_entry",
]

feature_cols = [c for c in DETECTION_FEATURES if c in df.columns]
reporting_cols = [c for c in REPORTING_FEATURES if c in df.columns]

print(f"Detection features ({len(feature_cols)})")
print(f"Reporting features ({len(reporting_cols)})")

Detection features (16)
Reporting features (11)


## 3.4 Scaling

IsolationForest and LOF are distance-sensitive. Without scaling, features with larger raw values (entries in the hundreds) would dominate features with smaller scales (rates between 0 and 1).

In [146]:
X = df[feature_cols].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Scaled matrix: {X_scaled.shape}")

Scaled matrix: (3267, 16)


## 3.5 Three detection methods

We apply three complementary unsupervised methods, each with `contamination=0.05`:

1. **Isolation Forest** — global multivariate outliers (joint feature space)
2. **Local Outlier Factor** — local density anomalies (neighborhood-based)
3. **Multivariate Z-score** — extreme single-feature deviations (most interpretable)

Fixing the same contamination across methods makes the comparison fair: each model has the same anomaly budget, so consensus voting reflects *which* observations they flag, not *how many*. Without it, the Z-score becomes far more aggressive than the others (33% vs 12%) and consensus becomes unbalanced.

In [147]:
iso = IsolationForest(contamination=CONTAMINATION, random_state=RANDOM_STATE, n_jobs=-1)
df["iso_label"] = iso.fit_predict(X_scaled)
df["iso_anomaly"] = (df["iso_label"] == -1).astype(int)
df["iso_score"] = iso.decision_function(X_scaled)
print(f"Isolation Forest: {df['iso_anomaly'].sum()} anomalies ({df['iso_anomaly'].mean():.1%})")

Isolation Forest: 164 anomalies (5.0%)


In [148]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=CONTAMINATION)
df["lof_label"] = lof.fit_predict(X_scaled)
df["lof_anomaly"] = (df["lof_label"] == -1).astype(int)
df["lof_score"] = lof.negative_outlier_factor_
print(f"LOF: {df['lof_anomaly'].sum()} anomalies ({df['lof_anomaly'].mean():.1%})")

LOF: 164 anomalies (5.0%)


In [149]:
df["zscore_max"] = np.abs(X_scaled).max(axis=1)
zscore_threshold = np.percentile(df["zscore_max"], 95)
df["zscore_anomaly"] = (df["zscore_max"] > zscore_threshold).astype(int)
print(f"Z-score: {df['zscore_anomaly'].sum()} anomalies ({df['zscore_anomaly'].mean():.1%})")

Z-score: 164 anomalies (5.0%)


### Consensus voting and overlap

Majority voting: an observation is flagged when at least 2 out of 3 methods agree. The pairwise overlap matrix shows whether the three methods are complementary or redundant.

In [150]:
df["anomaly_votes"] = df["iso_anomaly"] + df["lof_anomaly"] + df["zscore_anomaly"]
df["consensus_anomaly"] = (df["anomaly_votes"] >= 2).astype(int)
print(f"Consensus anomalies (>=2 votes): {df['consensus_anomaly'].sum()} ({df['consensus_anomaly'].mean():.1%})")

model_cols = ["iso_anomaly", "lof_anomaly", "zscore_anomaly"]
overlap = pd.DataFrame(index=model_cols, columns=model_cols, dtype=int)
for a in model_cols:
    for b in model_cols:
        overlap.loc[a, b] = int(((df[a] == 1) & (df[b] == 1)).sum())
print("\nPairwise overlap (count):")
print(overlap)

Consensus anomalies (>=2 votes): 108 (3.3%)

Pairwise overlap (count):
                iso_anomaly  lof_anomaly  zscore_anomaly
iso_anomaly           164.0          0.0           108.0
lof_anomaly             0.0        164.0             0.0
zscore_anomaly        108.0          0.0           164.0


### Feature importance

Extracted from the Isolation Forest, the only one of the three methods that exposes a native importance attribute (it's tree-based, so each feature gets a score based on how often and how early it is used to isolate observations).

In [151]:
importances = np.zeros(len(feature_cols))
for tree in iso.estimators_:
    importances += tree.feature_importances_
importances /= len(iso.estimators_)

feat_imp = pd.DataFrame({"feature": feature_cols, "importance": importances}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.head(15).plot.barh(x="feature", y="importance", ax=ax, legend=False, color="steelblue")
ax.invert_yaxis()
ax.set_title("Top 15 Features — Isolation Forest Importance"); ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "classical_feature_importance.png", dpi=150)
plt.show()

## 3.6 Sensitivity analysis

Without ground-truth labels we cannot measure precision or recall, but we can check whether the system is **stable**: does the consensus flagged set change drastically when contamination changes? A stable pipeline doesn't depend on a single arbitrary choice.

In [152]:
contamination_grid = [0.03, 0.05, 0.07, 0.10]
sensitivity_results = []

for c in contamination_grid:
    iso_t = IsolationForest(contamination=c, random_state=RANDOM_STATE, n_jobs=-1)
    iso_f = (iso_t.fit_predict(X_scaled) == -1).astype(int)
    lof_t = LocalOutlierFactor(n_neighbors=20, contamination=c)
    lof_f = (lof_t.fit_predict(X_scaled) == -1).astype(int)
    z_thr = np.percentile(np.abs(X_scaled).max(axis=1), 100 * (1 - c))
    z_f = (np.abs(X_scaled).max(axis=1) > z_thr).astype(int)
    consensus = ((iso_f + lof_f + z_f) >= 2).astype(int)
    sensitivity_results.append({
        "contamination": c,
        "iso_rate": round(iso_f.mean(), 4),
        "lof_rate": round(lof_f.mean(), 4),
        "zscore_rate": round(z_f.mean(), 4),
        "consensus_rate": round(consensus.mean(), 4),
        "consensus_count": int(consensus.sum())
    })

sensitivity_df = pd.DataFrame(sensitivity_results)
sensitivity_df.to_csv(CLASSICAL_DIR / "classical_sensitivity.csv", index=False)
print(sensitivity_df.to_string(index=False))

 contamination  iso_rate  lof_rate  zscore_rate  consensus_rate  consensus_count
          0.03    0.0300    0.0272       0.0300          0.0162               53
          0.05    0.0502    0.0502       0.0502          0.0331              108
          0.07    0.0701    0.0701       0.0701          0.0499              163
          0.10    0.1001    0.1001       0.1001          0.0759              248


## 3.7 Post-processing — business rules

Statistical models flag what's statistically unusual. Business rules add operational meaning. They run *after* detection on observable evidence an analyst would consider anyway, so it's fine for them to use reporting features (flag rate, alarm density).

Four rules:

- `flag_rate_dev_ratio7 > 3` → flag rate exceeded 3× recent baseline
- `entries_dev_ratio7 > 3` → passenger volume exceeded 3× recent baseline
- `abs(entries_residual_z) > 2` → entries deviate more than 2σ from monthly norm
- `alarm_density_per_entry > 95th percentile` → disproportionate alarm cases

In [153]:
# 3.7 Post-processing: causal business rules
# These rules are allowed to influence the final anomaly label because they rely on
# traffic behavior and historical baselines, not on post-event alarm information.

df["rule_entries_spike"] = (df["entries_dev_ratio7"] > 3).astype(int)
df["rule_high_residual_z"] = (df["entries_residual_z"].abs() > 2).astype(int)

if "entries_vs_route_mean" in df.columns:
    df["rule_above_route_average"] = (df["entries_vs_route_mean"] > 3).astype(int)
else:
    df["rule_above_route_average"] = 0

rule_cols_classical = [
    "rule_entries_spike",
    "rule_high_residual_z",
    "rule_above_route_average",
]

df["rule_count"] = df[rule_cols_classical].sum(axis=1)
df["rule_any"] = (df["rule_count"] > 0).astype(int)

print(f"Causal rules triggered: {df['rule_any'].sum()} / {len(df)}")
print(df[rule_cols_classical].sum().sort_values(ascending=False))


# Contextual signals
# These indicators are useful for explanation and prioritization, but they do not
# independently determine whether an observation is anomalous.

contextual_cols = []

if "flag_rate_dev_ratio7" in df.columns:
    df["context_flag_rate_spike"] = (df["flag_rate_dev_ratio7"] > 3).astype(int)
    contextual_cols.append("context_flag_rate_spike")

if "alarm_density_per_entry" in df.columns:
    alarm_95 = df["alarm_density_per_entry"].quantile(0.95)
    df["context_high_alarm_density"] = (df["alarm_density_per_entry"] > alarm_95).astype(int)
    contextual_cols.append("context_high_alarm_density")

if contextual_cols:
    df["contextual_signal_count"] = df[contextual_cols].sum(axis=1)
else:
    df["contextual_signal_count"] = 0

print(f"Contextual signals triggered: {(df['contextual_signal_count'] > 0).sum()} / {len(df)}")

if contextual_cols:
    print(df[contextual_cols].sum().sort_values(ascending=False))

Causal rules triggered: 151 / 3267
rule_high_residual_z        127
rule_above_route_average     47
rule_entries_spike           20
dtype: int64
Contextual signals triggered: 257 / 3267
context_flag_rate_spike       143
context_high_alarm_density    126
dtype: int64


### Final anomaly label

```
final_anomaly = consensus_anomaly OR rule_any
```

Captures both statistically unusual observations and operationally meaningful patterns. The result is an alert queue, not a definitive classification.

In [154]:
df["final_anomaly"] = ((df["consensus_anomaly"] == 1) | (df["rule_any"] == 1)).astype(int)
print(f"Final anomalies: {df['final_anomaly'].sum()} / {len(df)} ({df['final_anomaly'].mean():.1%})")

Final anomalies: 224 / 3267 (6.9%)


## 3.8 Risk score and review priority

The `risk_score` is a 0-100 operational severity index built from model agreement, rule strength, recent traffic spike and seasonal deviation. It is **not a calibrated probability of threat** — it's a ranking signal.

`review_priority` then divides flagged anomalies into actionable tiers (P1 immediate, P2 high, P3 standard, P4 monitor) using percentile thresholds within the flagged set. This guarantees a usable queue even if absolute scores are moderate.

In [155]:
# Clean infinities before scoring
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)

df["model_agreement"] = df[model_cols].sum(axis=1)

components = pd.DataFrame(index=df.index)
components["model_agreement"] = (df["model_agreement"] / len(model_cols)).clip(0, 1)
components["rule_strength"] = (df["rule_count"] / len(rule_cols_classical)).clip(0, 1)
components["entries_spike"] = robust_score(df["entries_dev_ratio7"])
components["seasonal_residual"] = robust_score(df["entries_residual_z"].abs())

weights = {"model_agreement": 0.40, "rule_strength": 0.25, "entries_spike": 0.20, "seasonal_residual": 0.15}
df["risk_score"] = sum(components[c] * w for c, w in weights.items()) * 100
df.loc[df["final_anomaly"] == 0, "risk_score"] *= 0.25
df["risk_score"] = df["risk_score"].round(2)

df["risk_level"] = pd.cut(df["risk_score"], bins=[-0.01, 20, 40, 60, 100],
                          labels=["Low", "Moderate", "High", "Critical"])

df["review_priority"] = "Not flagged"
flagged_mask = df["final_anomaly"] == 1
if flagged_mask.sum() > 0:
    fs = df.loc[flagged_mask, "risk_score"]
    q50, q80, q95 = fs.quantile([0.50, 0.80, 0.95])
    def assign_priority(s):
        if s >= q95: return "P1 - immediate review"
        if s >= q80: return "P2 - high priority"
        if s >= q50: return "P3 - standard review"
        return "P4 - monitor"
    df.loc[flagged_mask, "review_priority"] = fs.apply(assign_priority)

print("Risk-level distribution:")
print(df["risk_level"].value_counts().sort_index())
print("\nReview-priority among flagged:")
print(df.loc[flagged_mask, "review_priority"].value_counts())

Risk-level distribution:
risk_level
Low         3043
Moderate     120
High          77
Critical      27
Name: count, dtype: int64

Review-priority among flagged:
review_priority
P4 - monitor             112
P3 - standard review      67
P2 - high priority        32
P1 - immediate review     13
Name: count, dtype: int64


## 3.9 Evidence type and explanation

Each anomaly is labeled by where the alert came from: model consensus, rules, or both. This makes the system transparent — Reply or an analyst can see exactly why each observation made the queue.

In [156]:
rule_label_map = {
    "rule_entries_spike": "entries above recent baseline",
    "rule_high_residual_z": "seasonal residual unusually high",
    "rule_above_route_average": "entries above route historical average",
}

contextual_label_map = {
    "context_flag_rate_spike": "flag rate above recent baseline",
    "context_high_alarm_density": "alarm density in top 5%",
}

model_label_map = {
    "iso_anomaly": "Isolation Forest",
    "lof_anomaly": "LOF",
    "zscore_anomaly": "Z-score",
}


def evidence_type(row):
    model_flag = row["consensus_anomaly"] == 1
    rule_flag = row["rule_any"] == 1

    if model_flag and rule_flag:
        return "Model + causal rules"
    if model_flag:
        return "Model consensus only"
    if rule_flag:
        return "Causal business rules only"
    return "Not flagged"


def build_explanation(row):
    triggered_models = [
        label
        for col, label in model_label_map.items()
        if row.get(col, 0) == 1
    ]

    triggered_rules = [
        label
        for col, label in rule_label_map.items()
        if row.get(col, 0) == 1
    ]

    triggered_context = [
        label
        for col, label in contextual_label_map.items()
        if row.get(col, 0) == 1
    ]

    parts = []

    if triggered_models:
        parts.append("Models: " + ", ".join(triggered_models))

    if triggered_rules:
        parts.append("Causal rules: " + "; ".join(triggered_rules))

    if triggered_context:
        parts.append("Contextual evidence: " + "; ".join(triggered_context))

    return " | ".join(parts) if parts else "Not flagged"


df["evidence_type"] = df.apply(evidence_type, axis=1)
df["anomaly_explanation"] = df.apply(build_explanation, axis=1)

print(df.loc[df["final_anomaly"] == 1, "evidence_type"].value_counts())

evidence_type
Causal business rules only    116
Model consensus only           73
Model + causal rules           35
Name: count, dtype: int64


## 3.10 Ranked anomaly report and figures

In [157]:
anomaly_report = (df.loc[df["final_anomaly"] == 1]
                  .sort_values(["risk_score", "model_agreement", "rule_count"], ascending=False)
                  .reset_index(drop=True))

priority_cols = [
    "risk_score", "risk_level", "review_priority", "evidence_type", "anomaly_explanation",
    "consensus_anomaly", "model_agreement", "rule_any", "rule_count",
    "iso_anomaly", "lof_anomaly", "zscore_anomaly",
    "entries", "investigated", "flagged",
    "investigation_rate", "flag_rate",
    "entries_dev_ratio7", "entries_residual_z",
]
report_cols = [c for c in ["date", "route_airport", "route_city", "route_country"] + priority_cols
               if c in anomaly_report.columns]

print(f"Total final anomalies: {len(anomaly_report)}")
print(f"P1 immediate review: {(anomaly_report['review_priority'] == 'P1 - immediate review').sum()}")
display(anomaly_report[report_cols].head(20))

Total final anomalies: 224
P1 immediate review: 13


,date,route_airport,route_city,route_country,risk_score,risk_level,review_priority,evidence_type,anomaly_explanation,consensus_anomaly,...,iso_anomaly,lof_anomaly,zscore_anomaly,entries,investigated,flagged,investigation_rate,flag_rate,entries_dev_ratio7,entries_residual_z
0,2024-01-18,IST_FCO,ISTANBUL_ROMA,TUR_ITA,86.67,Critical,P1 - immediate review,Model + causal rules,"Models: Isolation Forest, Z-score | Causal rul...",1,...,1,0,1,10,10,0,1.000000,0.000000,3.076923,3.147534
1,2024-02-28,JED_FCO,JEDDAH_ROMA,SAU_ITA,86.67,Critical,P1 - immediate review,Model + causal rules,"Models: Isolation Forest, Z-score | Causal rul...",1,...,1,0,1,64,0,0,0.000000,0.000000,5.209302,4.495254
2,2024-02-20,KBL_BGY,KABUL_BERGAMO,AFG_ITA,86.67,Critical,P1 - immediate review,Model + causal rules,"Models: Isolation Forest, Z-score | Causal rul...",1,...,1,0,1,11,11,2,1.000000,0.181818,3.500000,4.023726
3,2024-01-18,SAW_BGY,ISTANBUL_BERGAMO,TR_ITA,86.67,Critical,P1 - immediate review,Model + causal rules,"Models: Isolation Forest, Z-score | Causal rul...",1,...,1,0,1,11,11,2,1.000000,0.181818,3.850000,3.199450
4,2024-02-24,SAW_FCO,ISTANBUL_ROMA,TUR_ITA,86.67,Critical,P1 - immediate review,Model + causal rules,"Models: Isolation Forest, Z-score | Causal rul...",1,...,1,0,1,12,12,1,1.000000,0.083333,3.652174,5.138809
5,2024-03-02,TIA_VRN,TIRANA_VERONA,ALB_ITA,86.67,Critical,P1 - immediate review,Model + causal rules,"Models: Isolation Forest, Z-score | Causal rul...",1,...,1,0,1,596,594,82,0.996644,0.137584,3.018813,2.985918
6,2024-01-25,IKA_FCO,TEHRAN_ROMA,IRN_ITA,84.61,Critical,P1 - immediate review,Model + causal rules,"Models: Isolation Forest, Z-score | Causal rul...",1,...,1,0,1,39,39,0,1.000000,0.000000,4.074627,2.363871
7,2024-02-25,RMF_MXP,MARSA ALAM_MILANO,EG_ITA,84.30,Critical,P1 - immediate review,Model + causal rules,"Models: Isolation Forest, Z-score | Causal rul...",1,...,1,0,1,17,16,10,0.941176,0.588235,3.238095,2.306026
8,2024-01-20,DXB_MXP,DUBAI_MILANO,ARE_ITA,78.33,Critical,P1 - immediate review,Model + causal rules,"Models: Isolation Forest, Z-score | Causal rul...",1,...,1,0,1,7,7,2,1.000000,0.285714,3.000000,4.473246
9,2024-02-22,IKA_FCO,TEHRAN_ROMA,IRN_ITA,78.33,Critical,P1 - immediate review,Model + causal rules,"Models: Isolation Forest, Z-score | Causal rul...",1,...,1,0,1,48,48,1,1.000000,0.020833,2.800000,2.828895


In [158]:
# Anomaly rate by detector + evidence breakdown
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rates = pd.Series({
    "Isolation Forest": df["iso_anomaly"].mean(),
    "LOF": df["lof_anomaly"].mean(),
    "Z-score": df["zscore_anomaly"].mean(),
    "Consensus": df["consensus_anomaly"].mean(),
    "Rules": df["rule_any"].mean(),
    "Final": df["final_anomaly"].mean(),
})
rates.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Classical: Anomaly Rate by Layer"); axes[0].set_ylabel("Rate")
axes[0].tick_params(axis="x", rotation=45)

evidence_dist = df.loc[df["final_anomaly"] == 1, "evidence_type"].value_counts()
evidence_dist.plot(kind="bar", ax=axes[1], color="tomato")
axes[1].set_title("Classical: Anomalies by Evidence Type"); axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "classical_anomaly_overview.png", dpi=150)
plt.show()

## 3.11 Stop classical timer + export

In [159]:
CLASSICAL_END = time.time()
CLASSICAL_TIME = round(CLASSICAL_END - CLASSICAL_START, 2)
print(f"Classical pipeline execution time: {CLASSICAL_TIME} seconds")

Classical pipeline execution time: 3.21 seconds


In [160]:
df.to_csv(CLASSICAL_DIR / "classical_full_dataset.csv", index=False)
anomaly_report[report_cols].to_csv(CLASSICAL_DIR / "classical_ranked_anomaly_report.csv", index=False)

classical_summary = {
    "pipeline": "classical",
    "execution_time_seconds": CLASSICAL_TIME,
    "total_observations": len(df),
    "iso_anomalies": int(df["iso_anomaly"].sum()),
    "lof_anomalies": int(df["lof_anomaly"].sum()),
    "zscore_anomalies": int(df["zscore_anomaly"].sum()),
    "consensus_anomalies": int(df["consensus_anomaly"].sum()),
    "rule_anomalies": int(df["rule_any"].sum()),
    "final_anomalies": int(df["final_anomaly"].sum()),
    "final_anomaly_rate": round(df["final_anomaly"].mean(), 4),
    "evidence_breakdown": df.loc[df["final_anomaly"] == 1, "evidence_type"].value_counts().to_dict(),
    "risk_level_breakdown": df.loc[df["final_anomaly"] == 1, "risk_level"].value_counts().to_dict(),
    "review_priority_breakdown": df.loc[df["final_anomaly"] == 1, "review_priority"].value_counts().to_dict(),
}

with open(CLASSICAL_DIR / "classical_summary.json", "w") as f:
    json.dump(classical_summary, f, indent=2, default=str)

print(json.dumps(classical_summary, indent=2, default=str))

{
  "pipeline": "classical",
  "execution_time_seconds": 3.21,
  "total_observations": 3267,
  "iso_anomalies": 164,
  "lof_anomalies": 164,
  "zscore_anomalies": 164,
  "consensus_anomalies": 108,
  "rule_anomalies": 151,
  "final_anomalies": 224,
  "final_anomaly_rate": 0.0686,
  "evidence_breakdown": {
    "Causal business rules only": 116,
    "Model consensus only": 73,
    "Model + causal rules": 35
  },
  "risk_level_breakdown": {
    "Moderate": 120,
    "High": 77,
    "Critical": 27,
    "Low": 0
  },
  "review_priority_breakdown": {
    "P4 - monitor": 112,
    "P3 - standard review": 67,
    "P2 - high priority": 32,
    "P1 - immediate review": 13
  }
}


## 3.12 Airport-specific classical report

The global classical report covers the whole network. The multi-agent pipeline below works on a single airport at a time, so for a fair comparison the classical logic is re-run on the same dynamically selected perimeter after the MAS run. This produces a `classical_<airport>_comparison_ready.csv` directly comparable with the multi-agent output.

In [161]:
def run_airport_classical(input_df, airport_code, out_dir):
    """Create a classical comparison subset for the selected airport.

    The airport is provided dynamically after the MAS run, so the classical
    and multi-agent outputs are always compared on the same perimeter.
    """
    airport_code = str(airport_code).upper().strip()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    d = input_df.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d = d.dropna(subset=["date"])
    d["route_airport"] = d["route_airport"].astype(str).str.upper()
    d = d[d["route_airport"].str.endswith(f"_{airport_code}")].copy()
    d = d.sort_values(["date", "route_airport"]).reset_index(drop=True)

    if len(d) < MIN_ROWS_FOR_DETECTION:
        print(f"Not enough data for {airport_code}: only {len(d)} rows")
        out_path = out_dir / f"classical_{airport_code}_comparison_ready.csv"
        pd.DataFrame(columns=[
            "arrival_airport", "date", "route_airport", "route_city", "route_country",
            "entries", "investigated", "flagged", "case_records", "flag_rate",
            "alarm_density_per_entry", "entries_dev_ratio7", "entries_residual_z",
            "iso_anomaly", "lof_anomaly", "zscore_anomaly", "consensus_anomaly",
            "rule_any", "final_anomaly", "evidence_type", "risk_score",
            "risk_level", "review_priority"
        ]).to_csv(out_path, index=False)
        return pd.DataFrame()

    comparison_cols = [
        "date", "route_airport", "route_city", "route_country",
        "entries", "investigated", "flagged",
        "case_records", "flag_rate", "alarm_density_per_entry",
        "entries_dev_ratio7", "entries_residual_z",
        "iso_anomaly", "lof_anomaly", "zscore_anomaly",
        "consensus_anomaly", "rule_any", "final_anomaly",
        "evidence_type", "risk_score", "risk_level", "review_priority",
    ]
    comparison_cols = [c for c in comparison_cols if c in d.columns]
    d["arrival_airport"] = airport_code

    out_path = out_dir / f"classical_{airport_code}_comparison_ready.csv"
    d[["arrival_airport"] + comparison_cols].to_csv(out_path, index=False)
    print(f"Saved {len(d)} rows for {airport_code} -> {out_path}")
    return d


# 4. Multi-Agent System

A LangGraph workflow with five agents, each running a locally-hosted Ollama model:

| Agent | Model | Role |
|-------|-------|------|
| Data Agent | `llama3.2:3b` | Extract IATA code, filter raw data |
| Baseline Agent | deterministic (no LLM) | Build baseline dataframe |
| Outlier Detection Agent | `phi4-mini-reasoning` | Run IsolationForest + LOF + Z-score, interpret |
| Risk Profiling Agent | qwen2.5:7b | Score risk, assign priorities |
| Report Agent | `llama3.2:3b` | Generate natural-language transit anomaly report |

**Design choice**: the Baseline Agent does not use an LLM. We tested an LLM-driven version and observed numerical hallucinations on baseline calculations. A deterministic tool guarantees accuracy — LLMs are kept for reasoning/interpretation, deterministic code for precise numerics. This is a practical engineering trade-off, validated with the supervisor.

## 4.1 Shared state

In [162]:
class AgentState(TypedDict, total=False):
    messages: Annotated[List[BaseMessage], operator.add]
    perimeter: str
    status: str
    reason: str
    passenger_json: str
    alarm_json: str
    baseline_dataframe_json: str
    anomaly_results: str
    scored_dataframe_json: str
    risk_profile: str
    final_report: str

## 4.2 Data Agent

Extracts the IATA code from the user's free-text request, filters the cleaned datasets, and hands structured JSON to the next agent. The LLM handles language understanding; a regex guard then ensures the perimeter actually appears in the user's input — preventing prompt injection from changing the airport.

In [163]:
def fetch_security_context(perimeter: str):
    p_up = perimeter.upper().strip()
    passenger = pd.read_csv(PASSENGERS_CLEAN)
    cases = pd.read_csv(CASES_CLEAN)

    passenger = passenger.drop(columns=["Unnamed: 0"], errors="ignore")
    cases = cases.drop(columns=["Unnamed: 0"], errors="ignore")

    passenger["departure_date"] = pd.to_datetime(passenger["departure_date"], errors="coerce")
    cases["departure_date"] = pd.to_datetime(cases["departure_date"], errors="coerce")
    passenger = passenger.dropna(subset=["departure_date"]).copy()
    cases = cases.dropna(subset=["departure_date"]).copy()

    # Filter perimeter
    p_mask = (passenger["arrival_airport_code"].astype(str).str.upper().eq(p_up)
              | passenger["arrival_city"].astype(str).str.upper().str.contains(p_up, na=False))
    c_mask = (cases["arrival_airport_code"].astype(str).str.upper().eq(p_up)
              | cases["arrival_city_name"].astype(str).str.upper().str.contains(p_up, na=False))
    p_df = passenger.loc[p_mask].copy()
    c_df = cases.loc[c_mask].copy()

    # Numeric cleanup + data-quality flags
    count_cols = ["passengers_entries_count", "passengers_investigated_count", "passengers_flagged_count"]
    p_df["negative_count_issue"] = False
    for col in count_cols:
        p_df[col] = pd.to_numeric(p_df[col], errors="coerce").fillna(0)
        p_df["negative_count_issue"] = p_df["negative_count_issue"] | (p_df[col] < 0)
        p_df[col] = p_df[col].clip(lower=0)
    if "total_flights" in c_df.columns:
        c_df["total_flights"] = pd.to_numeric(c_df["total_flights"], errors="coerce").fillna(0).clip(lower=0)

    p_df["logical_count_issue"] = (
        (p_df["passengers_flagged_count"] > p_df["passengers_investigated_count"])
        | (p_df["passengers_investigated_count"] > p_df["passengers_entries_count"]))
    p_df["data_quality_issue"] = (p_df["negative_count_issue"] | p_df["logical_count_issue"]).astype(int)

    # Route features
    p_df["date"] = p_df["departure_date"].dt.floor("D")
    c_df["date"] = c_df["departure_date"].dt.floor("D")
    p_df["route_airport"] = p_df["departure_airport_code"].astype(str) + "_" + p_df["arrival_airport_code"].astype(str)
    p_df["route_city"] = p_df["departure_city"].astype(str) + "_" + p_df["arrival_city"].astype(str)
    p_df["route_country"] = p_df["departure_country_code"].astype(str) + "_" + p_df["arrival_country_code"].astype(str)
    c_df["route_airport"] = c_df["departure_airport_code"].astype(str) + "_" + c_df["arrival_airport_code"].astype(str)
    c_df["route_city"] = c_df["departure_city_name"].astype(str) + "_" + c_df["arrival_city_name"].astype(str)
    c_df["route_country"] = c_df["departure_country_code"].astype(str) + "_" + c_df["arrival_country_code"].astype(str)

    print(f"DATA AGENT: {p_up} | passenger rows: {len(p_df)}, case rows: {len(c_df)}")
    return p_df, c_df


def data_agent_node(state):
    llm = ChatOllama(model="llama3.2:3b", temperature=0)
    user_request = state["messages"][-1].content

    prompt = f"""Extract the airport IATA code ONLY from the text between <USER_REQUEST> and </USER_REQUEST>.

<USER_REQUEST>
{user_request}
</USER_REQUEST>

Rules:
- Return ONLY one 3-letter uppercase IATA code explicitly present in USER_REQUEST.
- Do NOT use examples, do NOT invent or correct codes.
- If no 3-letter code appears, return NONE.

Answer only the code or NONE."""

    raw = llm.invoke([HumanMessage(content=prompt)]).content.strip().upper()

    # Deterministic safety: prefer codes from the actual user request
    user_matches = re.findall(r"\b[A-Z]{3}\b", user_request.upper())
    llm_matches = re.findall(r"\b[A-Z]{3}\b", raw)
    if user_matches:
        perimeter = user_matches[0]
    elif llm_matches:
        perimeter = llm_matches[0]
    else:
        return {
            "status": "blocked", "reason": "No valid IATA code found in input.",
            "perimeter": None, "passenger_json": "[]", "alarm_json": "[]",
            "messages": [HumanMessage(content="Data Agent blocked: no valid IATA code found.")]
        }

    p_df, c_df = fetch_security_context(perimeter)
    if p_df.empty:
        return {
            "status": "blocked", "reason": f"No passenger data for {perimeter}",
            "perimeter": perimeter, "passenger_json": "[]", "alarm_json": "[]",
            "messages": [HumanMessage(content=f"Data Agent blocked: no data for {perimeter}.")]
        }

    return {
        "status": "ok", "perimeter": perimeter,
        "passenger_json": p_df.to_json(orient="records", date_format="iso"),
        "alarm_json": c_df.to_json(orient="records", date_format="iso"),
        "messages": [HumanMessage(content=f"Data Agent: {perimeter} ({len(p_df)} records)")]
    }

## 4.3 Baseline Agent (deterministic)

Builds the route-day baseline dataframe directly in Python. No LLM is invoked here — the function `build_baseline_dataframe` is called as a tool by the agent node, which simply reports completion to the state.

In [164]:
def build_baseline_dataframe(passenger_data: str, alarm_data: str) -> pd.DataFrame:
    p_df = pd.read_json(io.StringIO(passenger_data), orient="records")
    c_df = pd.read_json(io.StringIO(alarm_data), orient="records")

    if p_df.empty:
        return pd.DataFrame()

    p_df["date"] = pd.to_datetime(p_df["date"], errors="coerce")
    p_df = p_df.dropna(subset=["date"])
    if not c_df.empty:
        c_df["date"] = pd.to_datetime(c_df["date"], errors="coerce")
        c_df = c_df.dropna(subset=["date"])

    # Base table
    rd = (p_df.groupby(["date", "route_airport"], as_index=False)
        .agg(route_city=("route_city", "first"),
             route_country=("route_country", "first"),
             entries=("passengers_entries_count", "sum"),
             investigations=("passengers_investigated_count", "sum"),
             flagged=("passengers_flagged_count", "sum"),
             data_quality_rows=("data_quality_issue", "sum"),
             nationality_count=("nationality", "nunique"),
             document_type_count=("document_type", "nunique"),
             airline_count=("airline", "nunique"),
             control_result_count=("control_result", "nunique")))

    # Cases
    if not c_df.empty:
        cd = (c_df.groupby(["date", "route_airport"], as_index=False)
              .agg(case_records=("event_type", "size"),
                   total_flights=("total_flights", "sum"),
                   unique_alarm_reasons=("alarm_reason", "nunique"),
                   unique_event_types=("event_type", "nunique")))
        rd = rd.merge(cd, on=["date", "route_airport"], how="left")
    for c in ["case_records", "total_flights", "unique_alarm_reasons", "unique_event_types"]:
        if c not in rd.columns:
            rd[c] = 0
        rd[c] = rd[c].fillna(0)

    # Calendar
    rd = rd.sort_values(["route_airport", "date"]).reset_index(drop=True)
    rd["weekday"] = rd["date"].dt.dayofweek
    rd["is_weekend"] = rd["weekday"].isin([5, 6]).astype(int)
    rd["month"] = rd["date"].dt.month
    rd["is_low_volume"] = (rd["entries"] < 10).astype(int)
    rd["is_low_volume_50"] = (rd["entries"] < 50).astype(int)

    # Rates
    rd["investigation_rate"] = np.where(rd["entries"] > 0, rd["investigations"] / rd["entries"], 0)
    rd["flag_rate"] = np.where(rd["entries"] > 0, rd["flagged"] / rd["entries"], 0)
    rd["alarm_density_per_entry"] = np.where(rd["entries"] > 0, rd["case_records"] / rd["entries"], 0)

    # Lags / diffs / pct_change
    for col in ["entries", "investigations", "flagged", "investigation_rate", "flag_rate"]:
        rd[f"{col}_lag1"] = rd.groupby("route_airport")[col].shift(1)
        rd[f"{col}_diff1"] = rd[col] - rd[f"{col}_lag1"]
        rd[f"{col}_pct_change1"] = np.where(
            rd[f"{col}_lag1"].notna() & (rd[f"{col}_lag1"] != 0),
            (rd[col] - rd[f"{col}_lag1"]) / rd[f"{col}_lag1"], 0)

    # Rolling and dev_ratio for entries, flag_rate, alarm_density
    for col in ["entries", "flag_rate", "alarm_density_per_entry"]:
        rd[f"{col}_roll7"] = rd.groupby("route_airport")[col].transform(lambda s: s.rolling(7, min_periods=3).mean())
        rd[f"{col}_roll30"] = rd.groupby("route_airport")[col].transform(lambda s: s.rolling(30, min_periods=7).mean())
        rd[f"{col}_dev_ratio7"] = np.where(
            rd[f"{col}_roll7"].notna() & (rd[f"{col}_roll7"] > 0),
            rd[col] / rd[f"{col}_roll7"], 0)

    # Monthly seasonal baseline + z-score
    rd["entries_monthly_baseline"] = rd.groupby(["route_airport", "month"])["entries"].transform("mean")
    rd["entries_residual"] = rd["entries"] - rd["entries_monthly_baseline"]
    res_std = rd.groupby("route_airport")["entries_residual"].transform("std")
    rd["entries_residual_z"] = np.where(res_std.notna() & (res_std > 0),
        rd["entries_residual"] / res_std, 0)
    rd["entries_route_mean"] = rd.groupby("route_airport")["entries"].transform("mean")
    rd["entries_vs_route_mean"] = np.where(rd["entries_route_mean"] > 0,
        rd["entries"] / rd["entries_route_mean"], 0)

    # Data-quality flag at route-day level
    rd["data_quality_issue"] = (rd["data_quality_rows"] > 0).astype(int)

    rd = rd.fillna(0)
    return rd


def baseline_agent_node(state):
    print("BASELINE AGENT: Preparing data hand-off")
    baseline_df = build_baseline_dataframe(state["passenger_json"], state["alarm_json"])
    return {
        "baseline_dataframe_json": baseline_df.to_json(orient="records", date_format="iso"),
        "messages": [HumanMessage(content=f"Baseline ready: {len(baseline_df)} route-day rows.")]
    }

## 4.4 Outlier Detection Agent

Runs the same three classical detectors (IF, LOF, Z-score) on the baseline dataframe — but now on a single airport. Consensus voting is applied, then five business rules (the four classical rules plus a `data_quality_issue` rule that's natural in the agent flow). The LLM then writes a short analytical note on the flagged rows. We strip `<think>` blocks emitted by `phi4-mini-reasoning` so the report stays clean.

In [165]:
DETECTION_FEATURES_MAS = [
    # Traffic and historical baseline features, aligned with the classical pipeline.
    "entries", "entries_lag1", "entries_diff1", "entries_pct_change1",
    "entries_roll7", "entries_roll30", "entries_dev_ratio7",
    "entries_monthly_baseline", "entries_residual", "entries_residual_z",
    "entries_route_mean", "entries_vs_route_mean",
    "weekday", "is_weekend", "is_low_volume", "is_low_volume_50",
    # Segment diversity signals available before post-event alarm outcomes.
    "nationality_count", "document_type_count", "airline_count", "control_result_count",
]


# Context/reporting variables such as flag rates, case records and alarm densities are
# intentionally excluded from DETECTION_FEATURES_MAS. They can enrich explanations,
# but they should not independently generate anomaly flags.


def outlier_detection_agent_node(state):
    print("OUTLIER DETECTION AGENT")

    baseline_json = state.get("baseline_dataframe_json", "[]")

    try:
        df = pd.read_json(io.StringIO(baseline_json), orient="records")
    except Exception as e:
        return {
            "anomaly_results": json.dumps({
                "error": f"Could not read baseline dataframe: {str(e)}"
            }),
            "scored_dataframe_json": "[]",
            "messages": [HumanMessage(content="Outlier Detection skipped: invalid baseline dataframe.")]
        }

    required_cols = ["date", "route_airport"]

    if df.empty or any(col not in df.columns for col in required_cols):
        available_cols = list(df.columns)

        return {
            "anomaly_results": json.dumps({
                "error": (
                    "No valid baseline data available for anomaly detection. "
                    "The selected perimeter may have no matching records, or the user input may not contain a valid IATA code."
                ),
                "rows": int(len(df)),
                "available_columns": available_cols
            }),
            "scored_dataframe_json": "[]",
            "messages": [HumanMessage(
                content=(
                    "Outlier Detection skipped: no valid baseline rows. "
                    "Try using an explicit 3-letter IATA code, e.g. TIA instead of Tirana."
                )
            )]
        }

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df = df.sort_values(["date", "route_airport"]).reset_index(drop=True)

    if len(df) < MIN_ROWS_FOR_DETECTION:
        return {
            "anomaly_results": json.dumps({
                "error": f"Only {len(df)} rows — not enough for detection."
            }),
            "scored_dataframe_json": "[]",
            "messages": [HumanMessage(content=f"Outlier Detection skipped: {len(df)} rows.")]
        }

    feature_cols = [c for c in DETECTION_FEATURES_MAS if c in df.columns]
    X = df[feature_cols].copy().replace([np.inf, -np.inf], np.nan)
    for col in X.columns:
        X[col] = X[col].fillna(X[col].median()) if not X[col].isna().all() else 0
    X_scaled = StandardScaler().fit_transform(X)

    # Three detectors — full dataset, no train/test split (parity with classical)
    iso = IsolationForest(contamination=CONTAMINATION, random_state=RANDOM_STATE, n_jobs=-1)
    df["iso_anomaly"] = (iso.fit_predict(X_scaled) == -1).astype(int)
    df["iso_score"] = iso.decision_function(X_scaled)

    n_neighbors = min(20, max(2, len(df) - 1))
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=CONTAMINATION)
    df["lof_anomaly"] = (lof.fit_predict(X_scaled) == -1).astype(int)
    df["lof_score"] = lof.negative_outlier_factor_

    zmax = np.abs(X_scaled).max(axis=1)
    z_threshold = np.percentile(zmax, 95)
    df["zscore_anomaly"] = (zmax > z_threshold).astype(int)
    df["zscore_max"] = zmax

    # Consensus
    df["anomaly_votes"] = df["iso_anomaly"] + df["lof_anomaly"] + df["zscore_anomaly"]
    df["consensus_anomaly"] = (df["anomaly_votes"] >= 2).astype(int)

    # Causal business rules, aligned with the classical pipeline.
    # These rules can influence the final anomaly label because they rely only on
    # traffic behavior and historical baselines.
    df["rule_entries_spike"] = (df["entries_dev_ratio7"] > ENTRIES_DEV_RATIO_THRESHOLD).astype(int)
    df["rule_high_residual_z"] = (df["entries_residual_z"].abs() > SEASONAL_Z_THRESHOLD).astype(int)
    if "entries_vs_route_mean" in df.columns:
        df["rule_above_route_average"] = (df["entries_vs_route_mean"] > ENTRIES_DEV_RATIO_THRESHOLD).astype(int)
    else:
        df["rule_above_route_average"] = 0

    rule_cols_mas = ["rule_entries_spike", "rule_high_residual_z", "rule_above_route_average"]
    df["rule_count"] = df[rule_cols_mas].sum(axis=1)
    df["rule_any"] = (df["rule_count"] > 0).astype(int)

    # Contextual evidence: useful for explanation and prioritisation, but not for
    # deciding final_anomaly on its own.
    contextual_cols_mas = []
    if "flag_rate_dev_ratio7" in df.columns:
        df["context_flag_rate_spike"] = (df["flag_rate_dev_ratio7"] > ENTRIES_DEV_RATIO_THRESHOLD).astype(int)
        contextual_cols_mas.append("context_flag_rate_spike")
    if "alarm_density_per_entry" in df.columns:
        alarm_95 = df["alarm_density_per_entry"].quantile(0.95)
        df["context_high_alarm_density"] = (df["alarm_density_per_entry"] > alarm_95).astype(int)
        contextual_cols_mas.append("context_high_alarm_density")
    if "data_quality_issue" in df.columns:
        df["context_data_quality_issue"] = (df["data_quality_issue"] == 1).astype(int)
        contextual_cols_mas.append("context_data_quality_issue")

    if contextual_cols_mas:
        df["contextual_signal_count"] = df[contextual_cols_mas].sum(axis=1)
    else:
        df["contextual_signal_count"] = 0

    df["final_anomaly"] = ((df["consensus_anomaly"] == 1) | (df["rule_any"] == 1)).astype(int)

    def evtype(r):
        m, ru = r["consensus_anomaly"] == 1, r["rule_any"] == 1
        if m and ru: return "Model + causal rules"
        if m: return "Model consensus only"
        if ru: return "Causal business rules only"
        return "Not flagged"
    df["evidence_type"] = df.apply(evtype, axis=1)

    def explain_row(r):
        models = [m for m, c in [("IsolationForest", "iso_anomaly"), ("LOF", "lof_anomaly"),
                                 ("Z-score", "zscore_anomaly")] if r.get(c, 0) == 1]
        causal = [c for c in rule_cols_mas if r.get(c, 0) == 1]
        context = [c for c in contextual_cols_mas if r.get(c, 0) == 1]
        parts = []
        if models:
            parts.append("Models: " + ", ".join(models))
        if causal:
            parts.append("Causal rules: " + ", ".join(causal))
        if context:
            parts.append("Contextual evidence: " + ", ".join(context))
        return " | ".join(parts) if parts else "Not flagged"

    df["anomaly_explanation"] = df.apply(explain_row, axis=1)

    # Compact output for LLM and downstream
    flagged = df[df["final_anomaly"] == 1].copy()
    flagged_rows = []
    for _, r in flagged.iterrows():
        methods = [m for m, c in [("IsolationForest", "iso_anomaly"), ("LOF", "lof_anomaly"),
                                  ("Z-score", "zscore_anomaly")] if r[c] == 1]
        triggered = [c for c in rule_cols_mas if r[c] == 1]
        contextual = [c for c in contextual_cols_mas if r.get(c, 0) == 1]
        flagged_rows.append({
            "date": str(r["date"].date()),
            "route_airport": r["route_airport"],
            "route_city": r["route_city"],
            "entries": int(r["entries"]),
            "investigations": int(r["investigations"]),
            "flagged": int(r["flagged"]),
            "case_records": int(r["case_records"]),
            "flag_rate": round(float(r["flag_rate"]), 4),
            "entries_dev_ratio7": round(float(r["entries_dev_ratio7"]), 2),
            "entries_residual_z": round(float(r["entries_residual_z"]), 2),
            "anomaly_votes": int(r["anomaly_votes"]),
            "flagged_by": methods,
            "triggered_rules": triggered,
            "contextual_evidence": contextual,
            "evidence_type": r["evidence_type"],
            "anomaly_explanation": r["anomaly_explanation"],
        })

    print(f"IF: {df['iso_anomaly'].sum()}, LOF: {df['lof_anomaly'].sum()}, "
          f"Z: {df['zscore_anomaly'].sum()}, Consensus: {df['consensus_anomaly'].sum()}, "
          f"Final: {df['final_anomaly'].sum()}")

    # LLM analytical note
    llm = ChatOllama(model="phi4-mini-reasoning", temperature=0)
    prompt = f"""Airport: {state["perimeter"]}
The Multi-Agent system analysed {len(df)} route-day observations.
Final flagged rows:
{json.dumps(flagged_rows[:20], indent=2)}

Write a short analytical note of maximum 6 sentences. Mention:
- whether anomalies are mostly model-driven, rule-driven, or both
- whether they are caused by route volume or historical-baseline deviations, and mention contextual alarm/case evidence only if present
- the most important dates/routes
Do not invent numbers. Do not mention suspicious nationalities or groups."""
    analysis = clean_llm_output(llm.invoke([HumanMessage(content=prompt)]).content)

    result = {
        "airport": state["perimeter"],
        "total_rows": int(len(df)),
        "feature_cols": feature_cols,
        "method_counts": {
            "IsolationForest": int(df["iso_anomaly"].sum()),
            "LOF": int(df["lof_anomaly"].sum()),
            "Z-score": int(df["zscore_anomaly"].sum())
        },
        "consensus_count": int(df["consensus_anomaly"].sum()),
        "business_rule_count": int(df["rule_any"].sum()),
        "final_anomaly_count": int(df["final_anomaly"].sum()),
        "zscore_threshold": round(float(z_threshold), 4),
        "flagged_rows": flagged_rows,
        "analysis": analysis
    }

    return {
        "anomaly_results": json.dumps(result),
        "scored_dataframe_json": df.to_json(orient="records", date_format="iso"),
        "messages": [HumanMessage(content=f"Outlier Detection: {result['final_anomaly_count']} final anomalies.")]
    }

## 4.5 Risk Profiling Agent

Computes the same risk score logic as the classical pipeline (model agreement, rule strength, traffic spike, seasonal residual) plus an alarm-density component. Assigns risk levels (LOW/MODERATE/HIGH/CRITICAL) and review priorities (P1-P4) to flagged rows. Uses `qwen2.5:7b` for the structured assessment generation.

In [166]:
from langchain_ollama import ChatOllama

qwen_risk_llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0,
    format="json"
)

In [167]:
def _safe_json_loads_from_llm(text):
    """
    Tries to parse JSON returned by the LLM.
    If the model adds extra text, it extracts the JSON object.
    """
    try:
        return json.loads(text)
    except Exception:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(text[start:end + 1])
        raise


def risk_profiling_agent_node(state):
    print("RISK PROFILING AGENT - QWEN")

    anomaly_data = json.loads(state["anomaly_results"])

    if "error" in anomaly_data:
        return {
            "risk_profile": json.dumps({"error": anomaly_data["error"]}),
            "messages": [HumanMessage(content="Risk profiling skipped.")]
        }

    df = pd.read_json(io.StringIO(state["scored_dataframe_json"]), orient="records")
    df["date"] = pd.to_datetime(df["date"])

    if df.empty:
        return {
            "risk_profile": json.dumps({
                "airport": state["perimeter"],
                "total_assessed": 0,
                "distribution": {},
                "assessments": []
            }),
            "messages": [HumanMessage(content="No rows to assess.")]
        }

    # -----------------------------
    # 1. Deterministic numeric layer
    # -----------------------------
    model_cols = ["iso_anomaly", "lof_anomaly", "zscore_anomaly"]

    rule_cols_mas = [
        "rule_entries_spike",
        "rule_high_residual_z",
        "rule_above_route_average",
    ]

    df["model_agreement"] = df[model_cols].sum(axis=1)

    components = pd.DataFrame(index=df.index)
    components["model_agreement"] = (df["model_agreement"] / len(model_cols)).clip(0, 1)
    components["rule_strength"] = (df[rule_cols_mas].sum(axis=1) / len(rule_cols_mas)).clip(0, 1)
    components["entries_spike"] = robust_score(df["entries_dev_ratio7"])
    components["seasonal_residual"] = robust_score(df["entries_residual_z"].abs())
    components["alarm_density"] = robust_score(df["alarm_density_per_entry_dev_ratio7"])

    weights = {
        "model_agreement": 0.35,
        "rule_strength": 0.25,
        "entries_spike": 0.15,
        "seasonal_residual": 0.15,
        "alarm_density": 0.10
    }

    df["risk_score"] = sum(components[c] * w for c, w in weights.items()) * 100

    # Penalize non-final anomalies, so they remain visible but low priority
    df.loc[df["final_anomaly"] == 0, "risk_score"] *= 0.25
    df["risk_score"] = df["risk_score"].round(2)

    df["risk_level"] = pd.cut(
        df["risk_score"],
        bins=[-0.01, 20, 40, 60, 100],
        labels=["LOW", "MODERATE", "HIGH", "CRITICAL"]
    ).astype(str)

    df["review_priority"] = "Not flagged"

    flagged_mask = df["final_anomaly"] == 1

    if flagged_mask.sum() > 0:
        fs = df.loc[flagged_mask, "risk_score"]
        q50, q80, q95 = fs.quantile([0.50, 0.80, 0.95])

        def assign_priority(score):
            if score >= q95:
                return "P1 - immediate review"
            if score >= q80:
                return "P2 - high priority"
            if score >= q50:
                return "P3 - standard review"
            return "P4 - monitor"

        df.loc[flagged_mask, "review_priority"] = fs.apply(assign_priority)

    # -----------------------------
    # 2. Prepare structured rows for Qwen
    # -----------------------------
    flagged_df = df.loc[flagged_mask].sort_values("risk_score", ascending=False).copy()

    assessments_input = []

    for _, r in flagged_df.iterrows():
        triggered = [c for c in rule_cols_mas if r.get(c, 0) == 1]

        if r.get("consensus_anomaly", 0) == 1:
            triggered.append("model_consensus")

        signals = {
            "model_agreement": int(r.get("model_agreement", 0)),
            "rule_entries_spike": int(r.get("rule_entries_spike", 0)),
            "rule_high_residual_z": int(r.get("rule_high_residual_z", 0)),
            "rule_above_route_average": int(r.get("rule_above_route_average", 0)),
            "rule_alarm_density_spike": int(r.get("rule_alarm_density_spike", 0)),
            "data_quality_issue": int(r.get("data_quality_issue", 0)),
            "entries_dev_ratio7": float(r.get("entries_dev_ratio7", 0)),
            "entries_residual_z": float(r.get("entries_residual_z", 0)),
            "alarm_density_per_entry_dev_ratio7": float(r.get("alarm_density_per_entry_dev_ratio7", 0)),
        }

        assessments_input.append({
            "date": str(r["date"].date()),
            "route_airport": r["route_airport"],
            "route_city": r["route_city"],
            "entries": float(r.get("entries", 0)),
            "flag_rate": float(r.get("flag_rate", 0)),
            "risk_score": float(r["risk_score"]),
            "risk_level": r["risk_level"],
            "review_priority": r["review_priority"],
            "evidence_type": r["evidence_type"],
            "triggered": triggered,
            "signals": signals
        })

    distribution = flagged_df["risk_level"].value_counts().to_dict()

    # -----------------------------
    # 3. Qwen risk interpretation layer
    # -----------------------------
    prompt = f"""
You are the Risk Profiling Agent in an airport transit anomaly detection system.

Your task is to produce a structured JSON risk profile.

Important rules:
- Do NOT change the numeric values.
- Do NOT recalculate risk_score.
- Do NOT change risk_level.
- Do NOT change review_priority.
- Use the provided evidence to write a short operational note for each anomaly.
- Notes must be concise, factual, and based only on the provided signals.
- Return valid JSON only.

Airport perimeter: {state["perimeter"]}

Input anomalies:
{json.dumps(assessments_input, indent=2)}

Return exactly this JSON structure:

{{
  "airport": "{state["perimeter"]}",
  "total_assessed": <number of anomalies>,
  "distribution": {{
    "LOW": <count if present>,
    "MODERATE": <count if present>,
    "HIGH": <count if present>,
    "CRITICAL": <count if present>
  }},
  "assessments": [
    {{
      "date": "...",
      "route_airport": "...",
      "route_city": "...",
      "risk_score": 0.0,
      "risk_level": "...",
      "review_priority": "...",
      "evidence_type": "...",
      "triggered": ["..."],
      "note": "..."
    }}
  ]
}}
"""

    try:
        llm_response = qwen_risk_llm.invoke(prompt)
        risk_profile = _safe_json_loads_from_llm(llm_response.content)

        # Safety correction: force key summary values from Python
        risk_profile["airport"] = state["perimeter"]
        risk_profile["total_assessed"] = len(assessments_input)
        risk_profile["distribution"] = distribution

    except Exception as e:
        print(f"Qwen risk profiling failed. Falling back to deterministic notes. Error: {e}")

        fallback_assessments = []

        for row in assessments_input:
            signals = row["signals"]

            notes = []

            if signals.get("data_quality_issue", 0) == 1:
                notes.append("contains a data-quality warning")
            if signals.get("rule_alarm_density_spike", 0) == 1:
                notes.append("alarm density above baseline")
            if signals.get("rule_entries_spike", 0) == 1:
                notes.append("entries above recent baseline")
            if signals.get("rule_high_residual_z", 0) == 1:
                notes.append("seasonal residual unusually high")
            if "model_consensus" in row["triggered"]:
                notes.append("confirmed by model consensus")

            if not notes:
                notes.append("flagged mainly by anomaly detection agreement")

            fallback_assessments.append({
                "date": row["date"],
                "route_airport": row["route_airport"],
                "route_city": row["route_city"],
                "risk_score": row["risk_score"],
                "risk_level": row["risk_level"],
                "review_priority": row["review_priority"],
                "evidence_type": row["evidence_type"],
                "triggered": row["triggered"],
                "note": "; ".join(notes) + "."
            })

        risk_profile = {
            "airport": state["perimeter"],
            "total_assessed": len(fallback_assessments),
            "distribution": distribution,
            "assessments": fallback_assessments
        }

    print(f"Risk distribution: {distribution}")

    return {
        "risk_profile": json.dumps(risk_profile),
        "scored_dataframe_json": df.to_json(orient="records", date_format="iso"),
        "messages": [HumanMessage(content=f"Qwen risk profile: {distribution}")]
    }

## 4.6 Report Agent

Produces the final transit anomaly report. Structured tables and metrics are computed deterministically in Python (so the LLM cannot invent numbers); the LLM only writes a short interpretation paragraph. Output is a Markdown document.

In [168]:
def report_agent_node(state):
    print("REPORT AGENT")
    anomaly_data = json.loads(state["anomaly_results"])
    risk_data = json.loads(state["risk_profile"])

    if "error" in anomaly_data or "error" in risk_data:
        airport = state.get("perimeter", "UNKNOWN")
        msg = anomaly_data.get("error") or risk_data.get("error")
        report = f"""# Transit Anomaly Report — {airport}

## Status
The report could not be generated because a previous step returned an error.

## Error
{msg}

## Interpretation
This usually means the selected airport has insufficient route-day observations for reliable anomaly detection.
"""
        return {"final_report": report, "messages": [HumanMessage(content="Report generation: error.")]}

    df = pd.read_json(io.StringIO(state["scored_dataframe_json"]), orient="records")
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date
    airport = state["perimeter"].upper().strip()

    flagged = df.loc[df["final_anomaly"] == 1].sort_values(
        ["risk_score", "model_agreement", "rule_count"], ascending=False).reset_index(drop=True)

    summary = {
        "airport": airport,
        "total_observations": int(len(df)),
        "final_anomalies": int(df["final_anomaly"].sum()),
        "final_anomaly_rate": round(float(df["final_anomaly"].mean()), 4),
        "model_consensus_anomalies": int(df["consensus_anomaly"].sum()),
        "business_rule_anomalies": int(df["rule_any"].sum()),
        "model_and_rule_overlap": int(((df["consensus_anomaly"] == 1) & (df["rule_any"] == 1)).sum()),
        "average_risk_score": round(float(df["risk_score"].mean()), 2),
    }

    top_anomalies = (flagged[["date", "route_airport", "route_city", "entries", "case_records",
                              "flag_rate", "risk_score", "risk_level", "review_priority",
                              "evidence_type"]].head(10) if len(flagged) else pd.DataFrame())

    # LLM interpretation note (short, deterministic facts only)
    interpretation = anomaly_data.get("analysis", "")
    try:
        llm = ChatOllama(model="llama3.2:3b", temperature=0)
        prompt = f"""You are the Report Agent. Write ONE short technical paragraph (max 5 sentences).
Use only these facts:
{json.dumps({"airport": airport, "summary": summary,
             "top_anomalies": top_anomalies.head(5).to_dict(orient='records') if len(top_anomalies) else []}, default=str)}

Do not invent numbers. Do not mention nationalities, ethnic groups, or political content."""
        llm_note = clean_llm_output(llm.invoke([HumanMessage(content=prompt)]).content)
        if llm_note:
            interpretation = llm_note
    except Exception as e:
        print(f"LLM note generation skipped: {e}")

    report = f"""# Transit Anomaly Report — {airport}

## Executive Summary
The Multi-Agent System analysed **{summary['total_observations']:,} route-day observations** for {airport} and flagged **{summary['final_anomalies']:,} final anomalies** ({summary['final_anomaly_rate']:.2%}).

This is an alert-prioritization output, not a supervised accuracy evaluation. No ground-truth labels are available, so precision/recall cannot be estimated.

## Summary Metrics
| Metric | Value |
|--------|-------|
| Total observations | {summary['total_observations']:,} |
| Final anomalies | {summary['final_anomalies']:,} |
| Final anomaly rate | {summary['final_anomaly_rate']:.2%} |
| Model consensus | {summary['model_consensus_anomalies']:,} |
| Business rules | {summary['business_rule_anomalies']:,} |
| Model + rules overlap | {summary['model_and_rule_overlap']:,} |
| Average risk score | {summary['average_risk_score']} |

## Top Anomalies
{top_anomalies.to_markdown(index=False) if len(top_anomalies) else "No flagged anomalies."}

## Interpretation
{interpretation}
"""
    return {"final_report": report, "messages": [HumanMessage(content="Report Agent: report generated.")]}

## 4.7 Build the graph

In [169]:
builder = StateGraph(AgentState)
builder.add_node("data_agent", data_agent_node)
builder.add_node("baseline_agent", baseline_agent_node)
builder.add_node("outlier_detection_agent", outlier_detection_agent_node)
builder.add_node("risk_profiling_agent", risk_profiling_agent_node)
builder.add_node("report_agent", report_agent_node)

builder.set_entry_point("data_agent")
builder.add_edge("data_agent", "baseline_agent")
builder.add_edge("baseline_agent", "outlier_detection_agent")
builder.add_edge("outlier_detection_agent", "risk_profiling_agent")
builder.add_edge("risk_profiling_agent", "report_agent")
builder.add_edge("report_agent", END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory)
print("Multi-Agent graph compiled.")

Multi-Agent graph compiled.


## 4.8 Run the Multi-Agent System

We start the **multi-agent timer** here. The user selects the target airport dynamically at runtime. The same airport is then used to generate the airport-specific classical subset, ensuring a fair comparison between the two pipelines.

In [170]:
available_airports = (
    df["route_airport"]
    .dropna()
    .astype(str)
    .str.upper()
    .str.extract(r"_([A-Z]{3})$")[0]
    .dropna()
    .sort_values()
    .unique()
)

print("Available airport examples:", ", ".join(available_airports[:30]))
user_q = input("Identify target airport/perimeter, e.g. FCO, MXP, LIN, NAP: ")

if not user_q.strip():
    raise ValueError("No airport/perimeter provided. Please enter a valid 3-letter IATA code.")

MAS_START = time.time()
config = {"configurable": {"thread_id": f"mas_run_{user_q.strip().upper()}"}}
final_state = None

for event in app.stream(
    {"messages": [HumanMessage(content=user_q)]},
    config,
    stream_mode="values"
):
    final_state = event

if final_state is None:
    raise RuntimeError("The multi-agent graph did not return a final state.")

MAS_END = time.time()
MAS_TIME = round(MAS_END - MAS_START, 2)
airport = final_state["perimeter"].upper().strip()

if airport not in available_airports:
    print(
        f"Warning: airport {airport} was not found in the route_airport codes. "
        "The run may produce zero rows. Available examples: " + ", ".join(available_airports[:20])
    )

print(f"\nSelected airport/perimeter: {airport}")
print(f"Multi-agent execution time: {MAS_TIME} seconds")


Available airport examples: AOI, BDS, BGY, BLQ, BRI, CAG, CIA, CIY, CTA, CUF, FCO, FLR, GOA, LIN, MXP, NAP, PEG, PMF, PMO, PSA, PSR, REG, RMI, SUF, TRN, TRS, TSF, VCE, VRN
DATA AGENT: FCO | passenger rows: 777, case rows: 1253
BASELINE AGENT: Preparing data hand-off
OUTLIER DETECTION AGENT
IF: 29, LOF: 29, Z: 26, Consensus: 21, Final: 40
RISK PROFILING AGENT - QWEN
Risk distribution: {'MODERATE': 26, 'HIGH': 9, 'CRITICAL': 5}
REPORT AGENT

Selected airport/perimeter: FCO
Multi-agent execution time: 134.54 seconds


In [171]:
print("=" * 70)
print(f"PERIMETER: {final_state.get('perimeter')}")
print("=" * 70)

print("\n--- OUTLIER DETECTION RESULTS ---")
anomaly = json.loads(final_state["anomaly_results"])
if "error" in anomaly:
    print(f"Error: {anomaly['error']}")
else:
    print(f"Total route-day rows: {anomaly['total_rows']}")
    print(f"Method counts: {anomaly['method_counts']}")
    print(f"Consensus: {anomaly['consensus_count']}, Rules: {anomaly['business_rule_count']}, Final: {anomaly['final_anomaly_count']}")
    print(f"\nAnalysis:\n{anomaly['analysis']}")

print("\n--- RISK PROFILING RESULTS ---")
risk = json.loads(final_state["risk_profile"])
if "error" in risk:
    print(f"Error: {risk['error']}")
else:
    print(f"Total assessed: {risk['total_assessed']}")
    print(f"Distribution: {risk['distribution']}")

print("\n--- FINAL TRANSIT ANOMALY REPORT ---")
print(final_state["final_report"][:1500])

PERIMETER: FCO

--- OUTLIER DETECTION RESULTS ---
Total route-day rows: 579
Method counts: {'IsolationForest': 29, 'LOF': 29, 'Z-score': 26}
Consensus: 21, Rules: 28, Final: 40

Analysis:
The anomalies in FCO airport routes are driven by both model-based (e.g., IsolationForest, Z-score) and rule-based systems (e.g., high residual_z deviations), with several cases involving combinations of these factors. Anomalies often stem from significant route volume spikes—such as the 353 entries on **2024-01-26 for TIRANA_ROMA**—or persistent historical-baseline deviations, like prolonged elevated residual_z values in low-volume scenarios (e.g., 2024-02-12 for HKT_FCO). Rule-driven flags frequently target high residual_z ratios tied to volume-to-historical baselines. Contextual alarm evidence is noted only sparingly: **ABUD DHABI on 2024-01-25** includes a contextual flag rate spike, while other dates report no additional context. Key anomalies also involve repeated triggers for TIRANA_ROMA across

## 4.9 Save MAS deliverables

In [172]:
airport = final_state["perimeter"].upper().strip()

# Generate the classical subset for the same dynamically selected airport.
classical_airport_dir = CLASSICAL_DIR / f"classical_{airport}"
classical_airport_dir.mkdir(parents=True, exist_ok=True)
classical_airport_df = run_airport_classical(df, airport, classical_airport_dir)

# Save the final report
report_path = AGENT_DIR / f"{airport}_transit_anomaly_report.md"
report_path.write_text(final_state["final_report"], encoding="utf-8")

# Save the scored route-day dataframe
scored_json = final_state.get("scored_dataframe_json", "[]")

try:
    scored_df = pd.read_json(io.StringIO(scored_json), orient="records")
except Exception as e:
    print(f"Could not read scored dataframe: {e}")
    scored_df = pd.DataFrame()

if scored_df.empty or "date" not in scored_df.columns:
    print(
        "No scored dataframe available for this MAS run. "
        "This usually means that the selected perimeter had no matching rows "
        "or not enough observations for anomaly detection."
    )

    scored_df = pd.DataFrame(columns=[
        "date",
        "arrival_airport",
        "route_airport",
        "route_city",
        "route_country",
        "entries",
        "investigated",
        "flagged",
        "consensus_anomaly",
        "final_anomaly",
        "risk_score",
        "risk_level",
        "review_priority",
        "evidence_type",
        "anomaly_explanation"
    ])
else:
    scored_df["date"] = pd.to_datetime(scored_df["date"], errors="coerce")
    scored_df["arrival_airport"] = airport

# Normalise naming for comparability with the classical output.
if "investigations" in scored_df.columns and "investigated" not in scored_df.columns:
    scored_df["investigated"] = scored_df["investigations"]

comparison_cols = [
    "date", "arrival_airport", "route_airport", "route_city",
    "entries", "investigated", "flagged", "case_records",
    "flag_rate", "alarm_density_per_entry",
    "entries_dev_ratio7", "entries_residual_z",
    "iso_anomaly", "lof_anomaly", "zscore_anomaly",
    "consensus_anomaly", "rule_any", "final_anomaly",
    "evidence_type", "risk_score", "risk_level", "review_priority",
]
comparison_cols = [c for c in comparison_cols if c in scored_df.columns]

scored_df.to_csv(AGENT_DIR / f"{airport}_agent_scored_route_day.csv", index=False)
scored_df[comparison_cols].to_csv(AGENT_DIR / f"{airport}_agent_comparison_ready.csv", index=False)

def safe_sum_column(df, col):
    if col in df.columns:
        return int(pd.to_numeric(df[col], errors="coerce").fillna(0).sum())
    return 0


def safe_mean_column(df, col):
    if col in df.columns and len(df) > 0:
        return float(pd.to_numeric(df[col], errors="coerce").fillna(0).mean())
    return 0.0


mas_summary = {
    "pipeline": "multi_agent",
    "airport": airport,
    "execution_time_seconds": MAS_TIME,
    "total_observations": int(len(scored_df)),
    "iso_anomalies": safe_sum_column(scored_df, "iso_anomaly"),
    "lof_anomalies": safe_sum_column(scored_df, "lof_anomaly"),
    "zscore_anomalies": safe_sum_column(scored_df, "zscore_anomaly"),
    "consensus_anomalies": safe_sum_column(scored_df, "consensus_anomaly"),
    "rule_anomalies": safe_sum_column(scored_df, "rule_any"),
    "final_anomalies": safe_sum_column(scored_df, "final_anomaly"),
    "final_anomaly_rate": round(safe_mean_column(scored_df, "final_anomaly"), 4),
    "status": "completed" if len(scored_df) > 0 and "final_anomaly" in scored_df.columns else "no_valid_scored_data",
    "error": None,
}

if "final_anomaly" in scored_df.columns:
    final_mask = scored_df["final_anomaly"] == 1
else:
    final_mask = pd.Series(False, index=scored_df.index)

mas_summary["evidence_breakdown"] = (
    scored_df.loc[final_mask, "evidence_type"].value_counts().to_dict()
    if "evidence_type" in scored_df.columns else {}
)
mas_summary["risk_level_breakdown"] = (
    scored_df.loc[final_mask, "risk_level"].value_counts().to_dict()
    if "risk_level" in scored_df.columns else {}
)
mas_summary["review_priority_breakdown"] = (
    scored_df.loc[final_mask, "review_priority"].value_counts().to_dict()
    if "review_priority" in scored_df.columns else {}
)
with open(AGENT_DIR / f"{airport}_agent_summary.json", "w") as f:
    json.dump(mas_summary, f, indent=2, default=str)

print(f"MAS deliverables saved in: {AGENT_DIR}")
print(json.dumps(mas_summary, indent=2, default=str))

Saved 579 rows for FCO -> c:\Users\AFGWO\OneDrive\Desktop\Lectures\Machine Learning\reply-classical-vs-multiagent\src\io\classical_report\classical_FCO\classical_FCO_comparison_ready.csv
MAS deliverables saved in: c:\Users\AFGWO\OneDrive\Desktop\Lectures\Machine Learning\reply-classical-vs-multiagent\src\io\agent_report
{
  "pipeline": "multi_agent",
  "airport": "FCO",
  "execution_time_seconds": 134.54,
  "total_observations": 579,
  "iso_anomalies": 29,
  "lof_anomalies": 29,
  "zscore_anomalies": 26,
  "consensus_anomalies": 21,
  "rule_anomalies": 28,
  "final_anomalies": 40,
  "final_anomaly_rate": 0.0691,
  "status": "completed",
  "error": null,
  "evidence_breakdown": {
    "Causal business rules only": 19,
    "Model consensus only": 12,
    "Model + causal rules": 9
  },
  "risk_level_breakdown": {
    "MODERATE": 26,
    "HIGH": 9,
    "CRITICAL": 5
  },
  "review_priority_breakdown": {
    "P4 - monitor": 20,
    "P3 - standard review": 12,
    "P2 - high priority": 6,
   

# 5. Comparative Analysis

Same airport, processed by both pipelines. We measure:

1. **Performance**: execution time of each pipeline
2. **Agreement**: how many anomalies overlap (Jaccard index)
3. **Coverage**: anomalies exclusive to each approach
4. **Output type**: structured CSV (classical) vs structured CSV + natural-language report (MAS)

Final deliverable: a `compared_report.md` that includes both timers side by side.

In [173]:
classical_path = CLASSICAL_DIR / f"classical_{airport}" / f"classical_{airport}_comparison_ready.csv"
agent_path = AGENT_DIR / f"{airport}_agent_comparison_ready.csv"

if not classical_path.exists():
    raise FileNotFoundError(f"Classical airport file missing: {classical_path}")
if not agent_path.exists():
    raise FileNotFoundError(f"Agent comparison file missing: {agent_path}")

cls_cmp = pd.read_csv(classical_path)
ag_cmp = pd.read_csv(agent_path)

# Normalize date strings only when the columns exist.
for cmp_df in [cls_cmp, ag_cmp]:
    if "date" in cmp_df.columns:
        cmp_df["date"] = pd.to_datetime(cmp_df["date"], errors="coerce").dt.strftime("%Y-%m-%d")

key = ["date", "route_airport"]

if all(c in cls_cmp.columns for c in key + ["final_anomaly"]):
    cls_anom = cls_cmp[cls_cmp["final_anomaly"] == 1].copy()
else:
    cls_anom = pd.DataFrame(columns=key)

if all(c in ag_cmp.columns for c in key + ["final_anomaly"]):
    ag_anom = ag_cmp[ag_cmp["final_anomaly"] == 1].copy()
else:
    ag_anom = pd.DataFrame(columns=key)

cls_keys = set(map(tuple, cls_anom[key].dropna().values)) if all(c in cls_anom.columns for c in key) else set()
ag_keys = set(map(tuple, ag_anom[key].dropna().values)) if all(c in ag_anom.columns for c in key) else set()
overlap_keys = cls_keys & ag_keys
union_keys = cls_keys | ag_keys

jaccard = len(overlap_keys) / max(len(union_keys), 1)
overlap_rate_cls = len(overlap_keys) / max(len(cls_keys), 1)
overlap_rate_ag = len(overlap_keys) / max(len(ag_keys), 1)

if len(cls_cmp) == 0 or len(ag_cmp) == 0:
    comparison_note = (
        "One of the two outputs contains zero rows, so agreement metrics should not "
        "be interpreted as a real model comparison."
    )
    print("Warning: one of the compared outputs has zero rows. Agreement metrics may not be meaningful.")
else:
    comparison_note = (
        "The Jaccard index measures agreement between the two flagged sets: 1.0 means "
        "perfect overlap, 0.0 means no shared anomalies."
    )

comparison_summary = pd.DataFrame({
    "metric": [
        "airport",
        "classical_rows", "agent_rows",
        "classical_anomalies", "agent_anomalies",
        "overlapping_anomalies",
        "jaccard_index",
        "overlap_rate_vs_classical", "overlap_rate_vs_agent",
        "classical_execution_seconds", "agent_execution_seconds",
        "speedup_classical_vs_agent"
    ],
    "value": [
        airport,
        len(cls_cmp), len(ag_cmp),
        len(cls_anom), len(ag_anom),
        len(overlap_keys),
        round(jaccard, 4),
        round(overlap_rate_cls, 4), round(overlap_rate_ag, 4),
        CLASSICAL_TIME, MAS_TIME,
        round(MAS_TIME / max(CLASSICAL_TIME, 0.001), 2)
    ]
})

comparison_summary.to_csv(COMPARED_DIR / f"comparison_summary_{airport}.csv", index=False)
display(comparison_summary)


,metric,value
0,airport,FCO
1,classical_rows,579
2,agent_rows,579
3,classical_anomalies,33
4,agent_anomalies,40
5,overlapping_anomalies,32
6,jaccard_index,0.7805
7,overlap_rate_vs_classical,0.9697
8,overlap_rate_vs_agent,0.8
9,classical_execution_seconds,3.21


## 5.1 Anomalies exclusive to each approach

In [174]:
only_cls = cls_anom[~cls_anom.set_index(key).index.isin(overlap_keys)].copy()
only_ag = ag_anom[~ag_anom.set_index(key).index.isin(overlap_keys)].copy()

print(f"Anomalies only in classical: {len(only_cls)}")
print(f"Anomalies only in multi-agent: {len(only_ag)}")
print(f"Overlapping: {len(overlap_keys)}")

if len(only_cls) > 0:
    print("\nTop classical-only:")
    display(only_cls.head(5))
if len(only_ag) > 0:
    print("\nTop agent-only:")
    display(only_ag.head(5))

Anomalies only in classical: 1
Anomalies only in multi-agent: 8
Overlapping: 32

Top classical-only:


,arrival_airport,date,route_airport,route_city,route_country,entries,investigated,flagged,case_records,flag_rate,...,iso_anomaly,lof_anomaly,zscore_anomaly,consensus_anomaly,rule_any,final_anomaly,evidence_type,risk_score,risk_level,review_priority
508,FCO,2024-08-02,TIA_FCO,TIRANA_ROMA,ALB_ITA,172,172,10,0.0,0.05814,...,1,0,1,1,0,1,Model consensus only,40.46,High,P3 - standard review



Top agent-only:


,date,arrival_airport,route_airport,route_city,entries,investigated,flagged,case_records,flag_rate,alarm_density_per_entry,...,iso_anomaly,lof_anomaly,zscore_anomaly,consensus_anomaly,rule_any,final_anomaly,evidence_type,risk_score,risk_level,review_priority
8,2024-01-01,FCO,TIA_FCO,TIRANA_ROMA,325,232,20,1,0.061538,0.003077,...,1,0,1,1,0,1,Model consensus only,33.57,MODERATE,P4 - monitor
242,2024-02-13,FCO,TIA_FCO,TIRANA_ROMA,235,221,14,0,0.059574,0.000000,...,1,0,1,1,0,1,Model consensus only,33.82,MODERATE,P4 - monitor
356,2024-02-24,FCO,TIA_FCO,TIRANA_ROMA,319,319,15,0,0.047022,0.000000,...,1,0,1,1,0,1,Model consensus only,42.05,HIGH,P3 - standard review
516,2024-09-01,FCO,TIA _FCO,TIRANA_ROMA,92,0,0,0,0.000000,0.000000,...,1,1,0,1,0,1,Model consensus only,29.20,MODERATE,P4 - monitor
545,2024-10-02,FCO,TIA_FCO,TIRANA_ROMA,392,343,17,1,0.043367,0.002551,...,1,0,1,1,0,1,Model consensus only,36.11,MODERATE,P3 - standard review


## 5.2 Compared report

The final document combines both summaries, the comparison metrics, the timing side by side, and the multi-agent's narrative report.

In [175]:
compared_report = f"""# Compared Report — Classical vs Multi-Agent

**Airport:** {airport} 
**Generated by:** Megaphone Telephone Gianluigi Buffon (Group 1)

## 1. Execution Time

| Pipeline | Time (seconds) |
|---|---|
| Classical | {CLASSICAL_TIME} |
| Multi-Agent | {MAS_TIME} |

The classical pipeline runs in deterministic Python and finishes in seconds. The multi-agent pipeline orchestrates four LLM calls plus deterministic computations, so it is significantly slower per run but produces interpreted output.

## 2. Anomaly Detection Summary

### Classical ({airport} subset)
- Total rows: **{len(cls_cmp)}**
- Final anomalies: **{len(cls_anom)}**
- Final anomaly rate: **{len(cls_anom) / max(len(cls_cmp), 1):.2%}**

### Multi-Agent ({airport})
- Total rows: **{len(ag_cmp)}**
- Final anomalies: **{len(ag_anom)}**
- Final anomaly rate: **{len(ag_anom) / max(len(ag_cmp), 1):.2%}**

## 3. Agreement Analysis

| Metric | Value |
|---|---|
| Overlapping anomalies | **{len(overlap_keys)}** |
| Jaccard index | **{jaccard:.4f}** |
| Overlap vs classical | **{overlap_rate_cls:.2%}** |
| Overlap vs multi-agent | **{overlap_rate_ag:.2%}** |
| Classical-only | **{len(only_cls)}** |
| Multi-Agent-only | **{len(only_ag)}** |

{comparison_note}

When both outputs contain rows, a partial overlap is expected and useful — it confirms the two pipelines capture related but not identical phenomena.

## 4. Output Difference

The classical pipeline produces a ranked CSV with anomaly scores, risk levels and rule flags. The multi-agent system produces the same structured output **plus** a natural-language transit anomaly report with operational interpretation.

For batch monitoring across the whole network, the classical pipeline is faster and reproducible. For interactive investigation of a specific perimeter, the multi-agent system adds a contextual narrative that a static CSV cannot deliver.

## 5. Multi-Agent Narrative Report

{final_state["final_report"]}

## 6. Limitations

- No ground-truth anomaly labels — both pipelines produce alert queues, not supervised classifications
- The multi-agent system works one airport at a time; scaling network-wide requires additional orchestration
- Both pipelines use configurable thresholds (contamination = {CONTAMINATION}, dev_ratio > {ENTRIES_DEV_RATIO_THRESHOLD}, |z| > {SEASONAL_Z_THRESHOLD}). Adaptive thresholds based on route-specific history are a natural next step
"""

compared_path = COMPARED_DIR / f"compared_report_{airport}.md"
compared_path.write_text(compared_report, encoding="utf-8")
print(f"Compared report saved to: {compared_path}")
print("\n" + "=" * 70)
print(compared_report[:2500])

Compared report saved to: c:\Users\AFGWO\OneDrive\Desktop\Lectures\Machine Learning\reply-classical-vs-multiagent\src\io\compared_report\compared_report_FCO.md

# Compared Report — Classical vs Multi-Agent

**Airport:** FCO 
**Generated by:** Megaphone Telephone Gianluigi Buffon (Group 1)

## 1. Execution Time

| Pipeline | Time (seconds) |
|---|---|
| Classical | 3.21 |
| Multi-Agent | 134.54 |

The classical pipeline runs in deterministic Python and finishes in seconds. The multi-agent pipeline orchestrates four LLM calls plus deterministic computations, so it is significantly slower per run but produces interpreted output.

## 2. Anomaly Detection Summary

### Classical (FCO subset)
- Total rows: **579**
- Final anomalies: **33**
- Final anomaly rate: **5.70%**

### Multi-Agent (FCO)
- Total rows: **579**
- Final anomalies: **40**
- Final anomaly rate: **6.91%**

## 3. Agreement Analysis

| Metric | Value |
|---|---|
| Overlapping anomalies | **32** |
| Jaccard index | **0.7805** |


## 5.3 Final summary

Pipeline timings, anomaly counts and the path to the compared report.

In [176]:
print(f"  CLASSICAL PIPELINE: {CLASSICAL_TIME}s — {classical_summary['final_anomalies']} anomalies on full dataset ({classical_summary['total_observations']} rows)")
print(f"  MULTI-AGENT SYSTEM:  {MAS_TIME}s — {mas_summary['final_anomalies']} anomalies on {airport} ({mas_summary['total_observations']} rows)")
print(f"  AGREEMENT ({airport}):    Jaccard {jaccard:.4f}, {len(overlap_keys)} overlapping anomalies")
print(f"  COMPARED REPORT:    {compared_path}")

  CLASSICAL PIPELINE: 3.21s — 224 anomalies on full dataset (3267 rows)
  MULTI-AGENT SYSTEM:  134.54s — 40 anomalies on FCO (579 rows)
  AGREEMENT (FCO):    Jaccard 0.7805, 32 overlapping anomalies
  COMPARED REPORT:    c:\Users\AFGWO\OneDrive\Desktop\Lectures\Machine Learning\reply-classical-vs-multiagent\src\io\compared_report\compared_report_FCO.md


# 6. Interactive Demo
 
A simple Gradio interface that runs both pipelines on a user-selected airport and displays the comparison live. Useful for the presentation: instead of showing static screenshots, the audience can see the actual workflow in real time.

In [177]:
import gradio as gr

In [178]:
def run_full_comparison(user_input):
    """
    Run the airport-specific classical and the multi-agent system on the same perimeter,
    then return four panels: timing, anomalies summary, comparison summary, MAS narrative.
    """
    if not user_input or not user_input.strip():
        return "No input provided.", "", "", ""
 
    user_input = user_input.strip()
 
    # ── 1. Run multi-agent system ──
    try:
        mas_start = time.time()
        config = {"configurable": {"thread_id": f"gradio_{user_input.upper()}"}}
        final_state = None
        for event in app.stream(
            {"messages": [HumanMessage(content=user_input)]},
            config,
            stream_mode="values"
        ):
            final_state = event
        mas_time = round(time.time() - mas_start, 2)
 
        if final_state is None or final_state.get("status") == "blocked":
            return (f"MAS blocked: {final_state.get('reason', 'unknown error') if final_state else 'no state'}",
                    "", "", "")
 
        airport = final_state["perimeter"].upper().strip()
    except Exception as e:
        return f"Multi-agent system failed: {e}", "", "", ""
 
    # ── 2. Run classical on the same airport ──
    try:
        classical_start = time.time()
        airport_dir = CLASSICAL_DIR / f"classical_{airport}"
        airport_dir.mkdir(parents=True, exist_ok=True)
        run_airport_classical(df, airport, airport_dir)
        classical_time = round(time.time() - classical_start, 2)
 
        cls_path = airport_dir / f"classical_{airport}_comparison_ready.csv"
        ag_path = AGENT_DIR / f"{airport}_agent_comparison_ready.csv"
 
        if not cls_path.exists() or not ag_path.exists():
            return f"Comparison files not found for {airport}.", "", "", ""
 
        cls_cmp = pd.read_csv(cls_path)
        ag_cmp = pd.read_csv(ag_path)
    except Exception as e:
        return f"Classical pipeline failed: {e}", "", "", ""
 
    # ── 3. Compute agreement metrics ──
    for d in [cls_cmp, ag_cmp]:
        if "date" in d.columns:
            d["date"] = pd.to_datetime(d["date"], errors="coerce").dt.strftime("%Y-%m-%d")
 
    key = ["date", "route_airport"]
    cls_anom = cls_cmp[cls_cmp["final_anomaly"] == 1] if "final_anomaly" in cls_cmp.columns else pd.DataFrame()
    ag_anom = ag_cmp[ag_cmp["final_anomaly"] == 1] if "final_anomaly" in ag_cmp.columns else pd.DataFrame()
 
    cls_keys = set(map(tuple, cls_anom[key].dropna().values)) if all(c in cls_anom.columns for c in key) else set()
    ag_keys = set(map(tuple, ag_anom[key].dropna().values)) if all(c in ag_anom.columns for c in key) else set()
    overlap_keys = cls_keys & ag_keys
    union_keys = cls_keys | ag_keys
    jaccard = len(overlap_keys) / max(len(union_keys), 1)
 
    # ── 4. Build the four output panels ──
    timing = f"""## Execution Time
 
| Pipeline | Time (seconds) |
|---|---|
| Classical (subset {airport}) | **{classical_time}** |
| Multi-Agent | **{mas_time}** |
 
The classical pipeline runs in deterministic Python. The multi-agent system orchestrates four LLM calls plus deterministic computations.
"""
 
    anomalies = f"""## Anomaly Detection
 
### Classical ({airport})
- Total rows: **{len(cls_cmp)}**
- Final anomalies: **{len(cls_anom)}**
- Anomaly rate: **{len(cls_anom) / max(len(cls_cmp), 1):.2%}**
 
### Multi-Agent ({airport})
- Total rows: **{len(ag_cmp)}**
- Final anomalies: **{len(ag_anom)}**
- Anomaly rate: **{len(ag_anom) / max(len(ag_cmp), 1):.2%}**
"""
 
    agreement = f"""## Agreement Analysis
 
| Metric | Value |
|---|---|
| Overlapping anomalies | **{len(overlap_keys)}** |
| Jaccard index | **{jaccard:.4f}** |
| Classical-only | **{len(cls_keys - ag_keys)}** |
| Multi-Agent-only | **{len(ag_keys - cls_keys)}** |
 
The Jaccard index measures agreement: 1.0 means perfect overlap, 0.0 means no shared anomalies. Partial overlap is expected — the two pipelines capture related but not identical phenomena.
"""
 
    mas_report = final_state.get("final_report", "Report not available.")
 
    return timing, anomalies, agreement, mas_report
 
 
# ── Cell 3 (code) ──
# Build the Gradio interface
with gr.Blocks(title="Reply × LUISS - Transit Anomaly Detection") as demo:
    gr.Markdown("""
# Reply × LUISS - Transit Anomaly Detection
## Classical Pipeline vs Multi-Agent System
 
Enter an airport code and run both pipelines side by side.
""")
 
    with gr.Row():
        airport_input = gr.Textbox(
            label="Target perimeter",
            placeholder="e.g. FCO, MXP, LIN, NAP",
            scale=4
        )
        run_btn = gr.Button("Run comparison", variant="primary", scale=1)
 
    with gr.Row():
        with gr.Column():
            timing_out = gr.Markdown(label="Timing")
        with gr.Column():
            anomalies_out = gr.Markdown(label="Anomalies")
 
    agreement_out = gr.Markdown(label="Agreement")
 
    gr.Markdown("---")
    gr.Markdown("## Multi-Agent Narrative Report")
    mas_out = gr.Markdown(label="MAS Report")
 
    run_btn.click(
        fn=run_full_comparison,
        inputs=airport_input,
        outputs=[timing_out, anomalies_out, agreement_out, mas_out]
    )
 
    gr.Examples(
        examples=["FCO", "MXP", "LIN", "NAP"],
        inputs=airport_input
    )
 
# Launch — set inline=False to open in a new browser tab; share=True to get a public URL
demo.launch(inline=True, share=False)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


DATA AGENT: FCO | passenger rows: 777, case rows: 1253
BASELINE AGENT: Preparing data hand-off
OUTLIER DETECTION AGENT
IF: 29, LOF: 29, Z: 26, Consensus: 21, Final: 40
RISK PROFILING AGENT - QWEN
Risk distribution: {'MODERATE': 26, 'HIGH': 9, 'CRITICAL': 5}
REPORT AGENT
Saved 579 rows for FCO -> c:\Users\AFGWO\OneDrive\Desktop\Lectures\Machine Learning\reply-classical-vs-multiagent\src\io\classical_report\classical_FCO\classical_FCO_comparison_ready.csv
